# KUL H02A5a Computer Vision: Group Assignment 2 (Group 19)

**Students:** Natalia Iglesias Bueno (r0877632), Carolina Marugan Rubio (r1072969), Merlijn Pietercil (r0852251), David Vargas Jorba (r1087132).


## Table of Contents

- [0. Setup and data loading](#0.-Setup-and-data-loading)
- [1. Classification](#1.-Classification)
  - [1.1 Data pipeline](#1.1-Data-pipeline)
  - [1.2 Loss function](#1.2-Loss-function)
  - [1.3 Metrics](#1.3-Metrics)
  - [1.4 Model A: from-scratch CNNs](#1.4-Model-A:-from-scratch-CNNs)
  - [1.5 Model B: transfer learning](#1.5-Model-B:-transfer-learning-with-ConvNeXt-Small-and-Vision-Transformer-(ImageNet-22k))
  - [1.6 Regularization](#1.6-Regularization)
  - [1.7 Training loop](#1.7-Training-loop)
  - [1.8 Evaluation and error analysis](#1.8-Evaluation-and-error-analysis)
  - [1.9 Test-set predictions](#1.9-Test-set-predictions)
- [2. Segmentation](#2.-Segmentation)
  - [2.1 Data pipeline](#2.1-Data-pipeline)
  - [2.2 Loss function](#2.2-Loss-function)
  - [2.3 Metrics](#2.3-Metrics)
  - [2.4 Model: DINOv2 + decoder](#2.4-Model:-DINOv2-ViT-B/14-(frozen)-+-multi-layer-concat-decoder)
  - [2.5 Regularization](#2.5-Regularization)
  - [2.6 Training loop](#2.6-Training-loop)
  - [2.7 Evaluation and error analysis](#2.7-Evaluation-and-error-analysis)
  - [2.8 Test-set predictions](#2.8-Test-set-predictions)
- [3. Adversarial attack](#3.-Adversarial-attack)
  - [3.1 Classification](#3.1-Classification)
  - [3.2 Segmentation](#3.2-Segmentation)
- [4. Discussion](#4-Discussion:)
- [References](#References)

In [ ]:
# Install packages not pre-installed on Kaggle.
# - iterative-stratification: multi-label stratified train/val split (Sechidis et al., ECML 2011)
# - timm: PyTorch image models, used for ConvNeXt and ViT with pretrained weights
# - segmentation-models-pytorch, albumentations: segmentation pipeline and image augmentations
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_ensure("iterative-stratification", "iterstrat")
_ensure("timm")
_ensure("segmentation-models-pytorch", "segmentation_models_pytorch")
_ensure("albumentations")

In [ ]:
import os, math, time, random, warnings, textwrap
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm, trange

from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import torchvision
from torchvision import transforms as T
from torchvision.models import resnet18

import timm
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoModel

## 0. Setup and data loading

This section configures global state and loads the dataset into pandas dataframes.


In [ ]:
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} on {DEVICE}")


In [ ]:
_KAGGLE_ROOT = Path("/kaggle/input/competitions/kul-computer-vision-ga-2-2026")

if _KAGGLE_ROOT.exists():
    DATA_ROOT = _KAGGLE_ROOT
else:
    DATA_ROOT = _LOCAL_ROOT
print(f"Using DATA_ROOT = {DATA_ROOT}")

train_df = pd.read_csv(DATA_ROOT / "train/train_set.csv", index_col="Id")
labels = list(train_df.columns)
NUM_CLASSES = len(labels)
print(f"{NUM_CLASSES} object classes: {labels}")

train_df["img"] = [np.load(DATA_ROOT / f"train/img/train_{idx}.npy") for idx, _ in train_df.iterrows()]
train_df["seg"] = [np.load(DATA_ROOT / f"train/seg/train_{idx}.npy") for idx, _ in train_df.iterrows()]
print(f"The training set contains {len(train_df)} examples.")

test_df = pd.read_csv(DATA_ROOT / "test/test_set.csv", index_col="Id")
test_df["img"] = [np.load(DATA_ROOT / f"test/img/test_{idx}.npy") for idx, _ in test_df.iterrows()]
test_df["seg"] = [-1 * np.ones(img.shape[:2], dtype=np.int8) for img in test_df["img"]]
print(f"The test set contains {len(test_df)} examples.")


## 0.1 Sanity check and class distribution

Before any model is built, we inspect the data. Three properties drive later design choices:

- class imbalance
- image size variability
- and the number of objects per image.

They influence whether to weight the loss, whether to resize aggressively, and whether the classifier head should produce a sigmoid output (one independent score per class) or a softmax output (one shared probability distribution).


In [ ]:
# Class balance for the multi-label classification task.
class_counts = train_df[labels].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 3))
class_counts.plot.bar(ax=ax)
ax.set_ylabel("# images containing this class")
ax.set_title("PASCAL VOC 2009 class frequency in training set")
plt.tight_layout()
plt.show()

# Average number of classes per image.
avg_labels = train_df[labels].sum(axis=1).mean()
print(f"Average labels per image: {avg_labels:.2f}")

# Image shape distribution (the dataset is not uniformly sized).
shapes = Counter(img.shape for img in train_df["img"])
print("Image shapes:", dict(list(shapes.items())[:10]))


In [ ]:
# Visualize a few images with their multi-label tags and segmentation maps.
def show_examples(df, n=6, seed=0):
    rng = np.random.RandomState(seed)
    idxs = rng.choice(len(df), size=n, replace=False)
    fig, axs = plt.subplots(2, n, figsize=(2.4 * n, 5))
    for j, k in enumerate(idxs):
        row = df.iloc[k]
        axs[0, j].imshow(row["img"]); axs[0, j].axis("off")
        active = [lab for lab in labels if row[lab] == 1]
        axs[0, j].set_title("\n".join(active), fontsize=8)
        axs[1, j].imshow(row["seg"], vmin=0, vmax=NUM_CLASSES, cmap="tab20")
        axs[1, j].axis("off")
    plt.tight_layout(); plt.show()

show_examples(train_df, n=6)


## 0.2 Kaggle submission helpers

In [ ]:
def _rle_encode(img):
    """
    Kaggle requires RLE encoded predictions for computation of the Dice score
    (https://www.kaggle.com/lifa08/run-length-encode-and-decode).

    Parameters
    ----------
    img: np.ndarray - binary img array

    Returns
    -------
    rle: str - running-length encoded version of img
    """
    pixels = img.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def generate_submission(df):
    """
    Convert the filled test dataframe into a Kaggle submission.csv with two rows per example
    (one for classification, one for segmentation), each RLE-encoded into a single Predicted column.

    Call this once, after both Sect. 1.9 (classification) and Sect. 2.8 (segmentation) predictions
    have been written into `test_df`.

    Parameters
    ----------
    df: pd.DataFrame - filled dataframe with binary label columns and an int seg map per row

    Returns
    -------
    submission_df: pd.DataFrame - df in Kaggle submission format (also written to submission.csv)
    """
    df_dict = {"Id": [], "Predicted": []}
    for idx, _ in df.iterrows():
        df_dict["Id"].append(f"{idx}_classification")
        df_dict["Predicted"].append(_rle_encode(np.array(df.loc[idx, labels])))
        df_dict["Id"].append(f"{idx}_segmentation")
        df_dict["Predicted"].append(
            _rle_encode(np.array([df.loc[idx, "seg"] == j + 1 for j in range(len(labels))]))
        )

    submission_df = pd.DataFrame(data=df_dict, dtype=str).set_index("Id")
    submission_df.to_csv("submission.csv")
    return submission_df

# 1. Classification

The task is to predict which of the 20 PASCAL VOC object classes are present in each image. Multiple classes can co-occur, so this is a *multi-label* problem, not a *multi-class* problem (mutually exclusive classes). This choice drives the loss function in Sect. 1.2. The training set contains 749 images, the test set 750.

Following the assignment, we trained two models per category to thoroughly compare approaches:

1. **Model A: from scratch.** Convolutional Neural Networks trained end-to-end on PASCAL VOC only, serving as our baselines without external data. We evaluate two variants:
   - **Model A1:** A compact, reduced-parameter wider ResNet-18 variant (~3M parameters).
   - **Model A2:** A standard ResNet-18 architecture (~11M parameters).
2. **Model B: transfer learning.** State-of-the-art architectures pretrained on ImageNet-21k/22k (which are independent of PASCAL VOC, satisfying the assignment's pretraining constraint), with the classifier heads replaced and fine-tuned on our training set. We evaluated two variants:
   - **Model B1:** ConvNeXt-Small (Liu et al., CVPR 2022), a modernized pure CNN pretrained on ImageNet-22k.
   - **Model B2:** Vision Transformer ViT-B/16 (Dosovitskiy et al., 2021) pretrained on ImageNet-21k.


## 1.1 Data pipeline

Choices informed by the exploratory analysis above:

- **Resize to 224 by 224.** ConvNeXt and the from-scratch CNN are designed for this input size. Matching it preserves filter receptive fields and normalization statistics.
- **ImageNet mean and standard deviation normalization.** Required by the pretrained backbone. We apply the same statistics to the from-scratch model so the two are comparable, since PASCAL VOC and ImageNet have similar color distributions.
- **Augmentation:** random horizontal flip, light color jitter (brightness, contrast, saturation), and `RandomResizedCrop(224, scale=(0.7, 1.0))`. Vertical flips and heavy geometric distortions are excluded because PASCAL VOC contains no upside-down objects.
- **Train/validation split: 80/20 with multi-label stratification** using `iterative-stratification` (Sechidis et al., ECML 2011). It distributes positives of every class proportionally across folds. With 749 images and several rare classes (under 40 instances), a random or single-class stratified split can leave a rare class with zero validation positives, making its per-class average precision undefined.

**Alternatives considered but rejected:**

- *Random or single-class stratified split:* rare classes can disappear from validation.
- *384 by 384 resolution:* higher memory and compute, no measurable accuracy gain in our tests.
- *No augmentation:* removes a standard regularizer.
- *K-fold cross-validation:* multiplies training time by K. We rely on the leaderboard for final ranking.

**Alternatives tested and rejected:**

- Data augmentation: We avoid vertical flips because PASCAL VOC contains no upside-down cars or people, and we avoid heavy geometric distortions. We did ablations of these flips as well as with `T.Resize((224, 224))`, which resizes the entire image, instead of `RandomResizedCrop(224, scale=(0.7,1.0))` which randomly resizes a crop of the image. The results were slightly better without the distortions and with the random crop.

In [ ]:
# PyTorch Dataset for multi-label classification on PASCAL VOC.
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class VOCClassificationDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=False)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = row["img"]                       # H x W x 3, uint8
        target = row[labels].values.astype(np.float32)  # 20-dim multi-hot
        # PIL conversion lets us use torchvision transforms directly.
        img = torchvision.transforms.functional.to_pil_image(img)
        if self.transform is not None:
            img = self.transform(img)
        return img, torch.from_numpy(target)


train_tf = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
# Multi-label stratified train and validation split.
# We use the iterative-stratification library, which implements the algorithm of
# Sechidis et al. (ECML 2011). It distributes the positives of every one of the
# 20 classes across the two folds in proportion, not just one chosen class.
y_multilabel = train_df[labels].values.astype(int)
splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, val_idx = next(splitter.split(np.arange(len(train_df)), y_multilabel))
train_split_df = train_df.iloc[tr_idx]
val_split_df = train_df.iloc[val_idx]
print(f"Train: {len(train_split_df)} | Val: {len(val_split_df)}")

# Sanity check: per-class positive counts in each fold.
counts = pd.DataFrame({
    "train": train_split_df[labels].sum(),
    "val": val_split_df[labels].sum(),
}).sort_values("train")
print(counts.to_string())



BATCH_SIZE = 32
NUM_WORKERS = 0

train_loader = DataLoader(
    VOCClassificationDataset(train_split_df, train_tf),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    VOCClassificationDataset(val_split_df, eval_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,
)

## 1.2 Loss function

Loss choices driven by the multi-label setup and the strong negative dominance (~18/20 classes negative per image, Sect. 0.1):

- **Per-class binary loss over categorical cross-entropy.** Categorical cross-entropy assumes mutually exclusive classes and does not apply to multi-label data. A *logit* is the raw network output before sigmoid or softmax.
- **Asymmetric Loss (ASL) over plain binary cross-entropy (BCE).** Plain BCE is dominated by easy negatives. ASL (Ben-Baruch et al., 2021) extends Focal Loss (Lin et al., 2017) with (1) decoupled focusing parameters for positives and negatives ($\gamma^- > \gamma^+$), and (2) a probability margin $m$ that zeroes the gradient from very confident negatives. The paper reports state-of-the-art on PASCAL VOC and MS-COCO. We use the paper's defaults: $\gamma^- = 4$, $\gamma^+ = 0$, $m = 0.05$.

For class $i$ with logit $z_i$, ground truth $y_i \in \{0, 1\}$, and $p_i = \sigma(z_i)$:

$$
\mathcal{L}_{ASL} = - \sum_{i=1}^{C}
\Big[
y_i (1 - p_i)^{\gamma_+} \log(p_i) + (1 - y_i)\, p_i^{\gamma_-} \log(1 - p_i)
\Big]
$$

**Alternatives considered.**

- *BCE with per-class `pos_weight`.* Standard imbalance fix that rescales the positive term only. ASL also down-weights easy negatives via $\gamma^-$ and the margin $m$.
- *Focal Loss* (Lin et al., 2017). ASL's predecessor with a single shared $\gamma$. Decoupled $\gamma^+/\gamma^-$ fits the strong negative dominance better.
- *Distribution-Balanced Loss* (Wu et al., 2020). Another long-tail multi-label loss. Not tested. Benchmark gap to ASL is small.

In [ ]:
# Asymmetric Loss for multi-label classification.
# Ben-Baruch et al., "Asymmetric Loss For Multi-Label Classification", ICCV 2021.
# Reference implementation: https://github.com/Alibaba-MIIL/ASL
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=0, clip=0.05, eps=1e-8, disable_torch_grad_focal_loss=True):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps
        self.disable_torch_grad_focal_loss = disable_torch_grad_focal_loss

    def forward(self, logits, targets):
        # logits: (B, C); targets: (B, C) in {0, 1}
        x_sigmoid = torch.sigmoid(logits)
        xs_pos = x_sigmoid
        xs_neg = 1 - x_sigmoid

        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        los_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1 - targets) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg

        if self.gamma_neg > 0 or self.gamma_pos > 0:
            # Save and restore the surrounding grad state. Setting grad_enabled
            # back to True unconditionally would corrupt evaluation contexts
            # wrapped in torch.no_grad().
            if self.disable_torch_grad_focal_loss:
                prev_grad = torch.is_grad_enabled()
                torch.set_grad_enabled(False)
            pt0 = xs_pos * targets
            pt1 = xs_neg * (1 - targets)
            pt = pt0 + pt1
            one_sided_gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
            one_sided_w = torch.pow(1 - pt, one_sided_gamma)
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(prev_grad)
            loss *= one_sided_w

        return -loss.sum() / logits.size(0)


classification_loss = AsymmetricLoss(gamma_neg=4, gamma_pos=0, clip=0.05)

## 1.3 Metrics

Metrics chosen for the multi-label setup and imbalanced class distribution (Sect. 0.1):

- **Mean Average Precision (mAP) as primary metric.** Follows the PASCAL VOC protocol (Everingham et al., 2010). mAP averages per-class area under the precision-recall curve. Threshold-free, so no decision cutoff is needed for evaluation.
- **Per-class F1 at a tuned threshold** as an interpretable secondary check. We pick the threshold that maximizes F1 per class on the validation set.

**Alternatives considered.**

- *Subset accuracy.* Counts an image as correct only if every label is predicted correctly. Treats one mislabeled class the same as misclassifying all twenty. Too coarse for 20-class multi-label data.
- *Hamming loss.* Averages the per-label error rate. An all-zero predictor scores low Hamming loss on a long-tail dataset without learning anything useful.
- *Macro-F1 at a fixed threshold of 0.5.* Forces a single decision cutoff. Rare classes whose optimal threshold lies below 0.5 are penalized. We report per-class tuned-threshold F1 instead.

In [ ]:
def compute_map(scores, targets):
    """mAP across classes. scores, targets: (N, C) numpy arrays."""
    aps = []
    for c in range(targets.shape[1]):
        if targets[:, c].sum() == 0:
            continue   # skip classes with no positives in this set
        aps.append(average_precision_score(targets[:, c], scores[:, c]))
    return float(np.mean(aps)), aps


def best_per_class_f1(scores, targets, thresholds=None):
    """For each class find the threshold that maximizes F1 on the given set."""
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19)
    best_thr = np.zeros(targets.shape[1])
    best_f1 = np.zeros(targets.shape[1])
    for c in range(targets.shape[1]):
        for t in thresholds:
            pred = (scores[:, c] >= t).astype(int)
            f1 = f1_score(targets[:, c], pred, zero_division=0)
            if f1 > best_f1[c]:
                best_f1[c] = f1
                best_thr[c] = t
    return best_thr, best_f1


## 1.4 Model A: from-scratch CNNs

Two ResNet-18 variants trained from random initialization on PASCAL VOC only, sharing loss (ASL), training loop, and data pipeline. The two variants differ only in channel width, so any validation mAP gap is attributable to capacity.

**Why ResNet over plain VGG.** ResNet (He et al., 2016) adds a *residual connection* that sums the input of a block with its output, so each block learns a correction on top of the previous representation rather than a full transformation. This solves the degradation problem (deeper plain networks training worse than shallow ones) and enables stable training at much greater depths than VGG-style networks (Simonyan and Zisserman, 2015). The deeper ResNet variants (50, 101, 152 layers) reach lower ImageNet top-5 error than VGG-19.

**Shared layer choices.**

- *Stem.* 7 by 7 convolution, stride 2, followed by Batch Normalization, ReLU, and 3 by 3 max-pool with stride 2.
- *Residual blocks.* Two 3 by 3 convolutions with Batch Normalization and ReLU (Ioffe and Szegedy, 2015), plus a skip connection from input to output.
- *Four stages.* Each stage doubles channels and halves spatial resolution. Within a stage, convolutions use stride 1, padding 1. The first convolution of each new stage uses stride 2.
- *Head.* Global average pooling, dropout (p=0.3), then a linear layer to 20 logits.
- *Initialization.* Kaiming scheme for ReLU networks (He et al., 2015).


### Model A1: reduced-width ResNet-18 (~3M parameters)

Channel widths halved at every stage, from (64, 128, 256, 512) to (32, 64, 128, 256), giving ~3M parameters. With only 749 training images, a smaller model has less capacity to overfit. The A1 vs A2 comparison isolates the effect of capacity under otherwise matched training.


### Model A2: Standard ResNet-18 (~11M parameters)

Standard ResNet-18 from `torchvision.models` with channel widths (64, 128, 256, 512), about 11M parameters. Direct A1/A2 comparison isolates the effect of capacity when training from scratch on a small dataset.

![Standard ResNet-18 architecture diagram](https://dl.acm.org/cms/attachment/html/10.1145/3613917.3613922/assets/html/images/icmssp2023-5-fig2.jpg)


In [ ]:
class BasicBlock(nn.Module):
    """ResNet BasicBlock (He et al., CVPR 2016).

    Two 3x3 convolutions with Batch Normalization and ReLU, plus an identity
    shortcut. When the stride or channel count changes, the shortcut becomes
    a 1x1 projection convolution.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


class SmallResNet(nn.Module):
    """Reduced-width ResNet-18 trained from scratch on PASCAL VOC.

    Channel widths are halved relative to the original ResNet-18, going from
    (64, 128, 256, 512) to (32, 64, 128, 256). Total parameters are about 3
    million, matched to the 749-image training set.
    """
    def __init__(self, num_classes=NUM_CLASSES,
                 channels=(32, 64, 128, 256),
                 blocks_per_stage=(2, 2, 2, 2)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels[0], 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(channels[0]),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        stages = []
        in_c = channels[0]
        for stage_idx, (out_c, n) in enumerate(zip(channels, blocks_per_stage)):
            blocks = []
            for b in range(n):
                stride = 2 if stage_idx > 0 and b == 0 else 1
                blocks.append(BasicBlock(in_c, out_c, stride=stride))
                in_c = out_c
            stages.append(nn.Sequential(*blocks))
        self.stages = nn.Sequential(*stages)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_c, num_classes))

        # Kaiming initialization (He et al., ICCV 2015) for ReLU networks.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        x = self.pool(x).flatten(1)
        return self.head(x)

## 1.5 Model B: Transfer Learning

Two transfer-learning baselines, both pretrained on ImageNet-21k (not PASCAL VOC) and fine-tuned on our training set, satisfying the assignment's pretraining constraint. Both share loss (ASL), training loop, and data pipeline.

**Why ImageNet-21k pre-training.** Ridnik et al. (2021) report consistent gains on downstream classification benchmarks compared to ImageNet-1k pre-training, with the largest gains on small downstream sets such as ours (749 images).

**Fine-tuning strategy (shared by B1 and B2).** Two stages:

1. *Warm-up.* Backbone frozen, only the new classifier head trained for a few epochs at a higher learning rate (~1e-3). Lets the head catch up before the backbone weights move.
2. *Full fine-tune.* Backbone unfrozen, smaller learning rate (~1e-4), cosine decay schedule.

`timm.create_model` replaces the original 1000-way classifier with a fresh `Linear` layer when `num_classes` is supplied. The training loop feeds raw logits to the loss.


### Model B1: ConvNeXt-Small

ConvNeXt-Small (Liu et al., 2022), pretrained on ImageNet-21k and fine-tuned on ImageNet-1k (timm checkpoint `convnext_small.fb_in22k_ft_in1k`). ConvNeXt is a pure convolutional network descended from ResNet: it keeps residual connections and the four-stage design, with larger 7 by 7 depthwise convolutions, GELU activations, and Layer Normalization instead of Batch Normalization. Liu et al. report 84.6 percent top-1 on ImageNet-1k with the 21k-pretrained variant, comparable to Swin-Small.


### Model B2: Vision Transformer (ViT-B/16)

ViT-B/16 (Dosovitskiy et al., 2021), pretrained on ImageNet-21k and fine-tuned on ImageNet-1k (timm checkpoint `vit_base_patch16_224.augreg_in21k_ft_in1k`). ViT splits each image into 16 by 16 patches, embeds each patch, prepends a learnable `[CLS]` token, adds positional embeddings, and processes the sequence through a standard Transformer encoder. The final `[CLS]` representation is used for classification.

![ViT architecture diagram](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/vit_architecture.jpg)


**Alternatives considered.**

- *ResNet-50 with ImageNet-1k pretraining.* Classical baseline (He et al., 2016). Smaller pretraining corpus than ImageNet-21k. Ridnik et al. (2021) show 21k pretraining transfers better.
- *ConvNeXt-Base / Large.* Strictly more parameters and FLOPs at the same input size. Heavier to train, no expected gain at our data scale.
- *Larger ViTs (ViT-L/16, ViT-H/14).* Same trade-off as ConvNeXt-Base/Large.


In [ ]:
def build_convnext_classifier(num_classes=NUM_CLASSES, freeze_backbone=True,
                              model_name="convnext_small.fb_in22k_ft_in1k"):
    """ConvNeXt-Small (Liu et al., CVPR 2022) pre-trained on ImageNet-22k.

    The 22k checkpoint provides the best transfer-learning starting point on
    small multi-label datasets such as PASCAL VOC (Ridnik et al., 2021).
    timm.create_model replaces the classifier head automatically when
    num_classes is supplied.
    """
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
        for p in model.get_classifier().parameters():
            p.requires_grad = True
    return model


def freeze_bn(model):
    """Keep BatchNorm in eval mode and stop updating its running statistics. Useful on small datasets.

    Has no effect on networks that do not use BatchNorm, such as ConvNeXt
    (which uses LayerNorm).
    """
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False

## 1.6 Regularization

With 749 training images and several million trainable parameters, overfitting is the main concern. We combine:

- **Decoupled weight decay** (Loshchilov and Hutter, 2019). Adds an L2 penalty $\lambda \sum_i w_i^2$ on the weights, applied directly to parameters rather than folded into the gradient. Weight decay is tuned per model (1e-4 to 1e-3 range, listed in Sect. 1.7).
- **Dropout (p=0.3) in the classifier head of Model A** (Srivastava et al., 2014). Randomly zeros activations during training, which discourages co-adaptation between units.
- **Batch Normalization in Model A** (cited in Sect. 1.4). Acts as a mild regularizer because each example sees slightly different normalization statistics at every step. Model B uses Layer Normalization instead (per-example, batch-size independent).
- **Data augmentation** (Sect. 1.1). Horizontal flips, light color jitter, and random resized crops act as a label-preserving distribution over the training set, multiplying the effective number of distinct training examples.


## 1.7 Training loop

`train_classifier` (next cell) implements the training loop: AdamW (with the weight decay from Sect. 1.6), cosine annealing learning-rate schedule, mixed-precision forward and backward when on CUDA, per-epoch validation, and best-checkpoint selection on validation mAP. Mini-batch size is 32. The function returns the model with the best weights loaded plus the per-epoch loss and mAP history.

Per-model hyperparameters:

| Model | Epochs | LR | Weight decay | Notes |
|---|---|---|---|---|
| A1 | 100 | 1e-3 | 5e-4 | From scratch |
| A2 | 100 | 1e-3 | 8e-4 | From scratch |
| B1 warm-up | 5 | 1e-3 | 1e-4 | Backbone frozen |
| B1 fine-tune | 20 | 1e-4 | 1e-4 | Full network |
| B2 warm-up | 5 | 5e-3 | 1e-3 | Backbone frozen |
| B2 fine-tune | 30 | 3e-5 | 1e-3 | Full network |


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Run a forward pass over a loader and return loss, mAP, per-class AP, scores, targets."""
    model.eval()
    all_scores, all_targets = [], []
    total_loss, n = 0.0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y_dev = y.to(DEVICE, non_blocking=True)
        logits = model(x)
        loss = classification_loss(logits, y_dev)
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
        all_scores.append(torch.sigmoid(logits).detach().cpu().numpy())
        all_targets.append(y.numpy())
    scores = np.concatenate(all_scores, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    map_score, aps = compute_map(scores, targets)
    val_loss = total_loss / max(n, 1)
    return val_loss, map_score, aps, scores, targets


def train_classifier(model, train_loader, val_loader, *, epochs, lr, weight_decay=1e-4,
                     freeze_bn_layers=False, log_every=1):
    model = model.to(DEVICE)
    if freeze_bn_layers:
        freeze_bn(model)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler(DEVICE.type, enabled=(DEVICE.type == "cuda"))

    history = {"train_loss": [], "val_loss": [], "val_map": []}
    best_map, best_state = -1.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        if freeze_bn_layers:
            freeze_bn(model)
        running, n = 0.0, 0
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast(DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits = model(x)
                loss = classification_loss(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * x.size(0)
            n += x.size(0)
        scheduler.step()
        train_loss = running / max(n, 1)

        val_loss, val_map, _, _, _ = evaluate(model, val_loader)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_map"].append(val_map)

        if val_map > best_map:
            best_map = val_map
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch % log_every == 0 or epoch == epochs:
            print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val mAP {val_map:.4f} | best {best_map:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


def plot_history(history, title=""):
    """Plot training loss, validation loss, and validation mAP across epochs.

    The two loss curves are the main tool for diagnosing learning rate.
    Validation mAP is the metric we track for checkpoint selection.
    """
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="train loss", color="tab:blue")
    axes[0].plot(epochs, history["val_loss"], label="val loss", color="tab:orange")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[1].plot(epochs, history["val_map"], color="tab:green")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("validation mAP")
    axes[1].set_title("Validation mAP")
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    plt.show()


### Model A1
SmallResNet from random initialization, 100 epochs.


In [ ]:
# Train Model A1: reduced-parameter wider ResNet-18 from scratch.
model_a1 = SmallResNet().to(DEVICE)
model_a1, hist_a1 = train_classifier(
    model_a1, train_loader, val_loader,
    epochs=100, lr=1e-3, weight_decay=5e-4,
)


In [ ]:
plot_history(hist_a1, title="Model A1: SmallResNet (from scratch)")


### Model A2
torchvision ResNet-18 from random initialization, 100 epochs.


In [ ]:
def build_standard_resnet18(num_classes=NUM_CLASSES):
    """Standard ResNet-18 from torchvision, trained from scratch.

    Uses the full channel widths (64, 128, 256, 512), about 11M parameters.
    The final fully connected layer is replaced with Dropout(0.3) + Linear so
    the head structure matches Model A1's SmallResNet.
    """
    from torchvision.models import resnet18
    model = resnet18(weights=None)  # no pre-trained weights, train from scratch
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    return model


model_a2_test = build_standard_resnet18()
print(f"Model A2 (Standard ResNet-18) parameters: {sum(p.numel() for p in model_a2_test.parameters()):,}")
del model_a2_test


In [ ]:
# Train Model A2: standard ResNet-18 from scratch.
# Same loss (AsymmetricLoss), same training loop, same data pipeline as Model A1.
# Weight decay is 8e-4 (higher than A1's 5e-4) to push back on the larger model's
# greater overfitting tendency on 749 images.
model_a2 = build_standard_resnet18().to(DEVICE)
model_a2, hist_a2 = train_classifier(
    model_a2, train_loader, val_loader,
    epochs=100, lr=1e-3, weight_decay=8e-4,
)


In [ ]:
plot_history(hist_a2, title="Model A2: Standard ResNet-18 (from scratch)")


### Model B1
ConvNeXt-Small with two-stage fine-tuning (5 epochs warm-up + 20 epochs full).


In [ ]:
# Build Model B1 (ConvNeXt-Small) and warm up the new classifier head with the backbone frozen.
model_tl = build_convnext_classifier(freeze_backbone=True).to(DEVICE)
model_tl, hist_warm = train_classifier(
    model_tl, train_loader, val_loader,
    epochs=5, lr=1e-3, weight_decay=1e-4,
)

In [ ]:
plot_history(hist_warm, title="Model B1 warm-up (head only)")

In [ ]:
# Unfreeze the backbone and fine-tune the whole model at a smaller learning rate.
for p in model_tl.parameters():
    p.requires_grad = True
model_tl, hist_ft = train_classifier(
    model_tl, train_loader, val_loader,
    epochs=20, lr=1e-4, weight_decay=1e-4,
)


In [ ]:
plot_history(hist_ft, title="Model B1 full fine-tune")

### Model B2
ViT-B/16 with two-stage fine-tuning (5 epochs warm-up + 30 epochs full).


In [ ]:
def build_vit_classifier(num_classes=NUM_CLASSES, freeze_backbone=True):
    """ViT-B/16 pre-trained on ImageNet-22k (Dosovitskiy et al., ICLR 2021).

    Uses timm's ViT-B/16 with ImageNet-22k weights (vit_base_patch16_224.augreg_in21k_ft_in1k).
    The classification head is replaced automatically by timm when `num_classes` is supplied.
    When `freeze_backbone=True`, only the head parameters are trainable.
    """
    import timm
    model = timm.create_model("vit_base_patch16_224.augreg_in21k_ft_in1k", pretrained=True, num_classes=num_classes)
    if freeze_backbone:
        for name, p in model.named_parameters():
            if not name.startswith("head"):
                p.requires_grad = False
    return model


model_vit_test = build_vit_classifier(freeze_backbone=False)
print(f"Model B2 (ViT-B/16) parameters: {sum(p.numel() for p in model_vit_test.parameters()):,}")
del model_vit_test

In [ ]:
# Build Model B2 (ViT-B/16) and warm up the new classifier head with the backbone frozen.
model_vit = build_vit_classifier(freeze_backbone=True).to(DEVICE)
model_vit, hist_vit_warm = train_classifier(
    model_vit, train_loader, val_loader,
    epochs=5, lr=5e-3, weight_decay=1e-3,
    freeze_bn_layers=True,
)


In [ ]:
plot_history(hist_vit_warm, title="Model B2 warm-up (head only)")


In [ ]:
# Unfreeze the ViT backbone and fine-tune the whole model at a smaller learning rate.
for p in model_vit.parameters():
    p.requires_grad = True
model_vit, hist_vit_ft = train_classifier(
    model_vit, train_loader, val_loader,
    epochs=30, lr=3e-5, weight_decay=1e-3,
    freeze_bn_layers=False,
)

In [ ]:
plot_history(hist_vit_ft, title="Model B2 full fine-tune")


### Model comparison summary

Four classifiers compared, all using ASL, the same data pipeline, and best-checkpoint selection on validation mAP.

- **A1 vs A2** isolates the effect of model capacity when training from scratch. Both share architecture style (ResNet basic blocks), head structure (dropout p=0.3 + linear), and training recipe. Weight decay is tuned per model (5e-4 for A1, 8e-4 for A2) to compensate for the larger model's greater overfitting tendency.
- **B1 vs B2** compares a modern CNN (ConvNeXt, with strong local inductive biases) against a pure Transformer (ViT, with global self-attention) at similar 21k → 1k pretraining. Hyperparameters were tuned per model rather than held equal.
- **A vs B** shows the benefit of transfer learning from ImageNet-21k.

**Hyperparameter tuning notes.**

- *B2 fine-tune learning rate.* We initially used LR=3e-6 for the ViT fine-tune, which was too low. Raising it to 3e-5 lifted the best mAP from 0.77 to 0.887.
- *B2 pretraining choice.* With ImageNet-1k pretraining (instead of 21k) and adjusted head LRs, we reached the same best mAP. The 21k variant has flatter loss curves. We kept 21k for consistency with B1.


## 1.8 Evaluation and error analysis

For each trained model we produce:

- **Validation report** with mAP, per-class AP, and per-class F1 at tuned thresholds (metrics defined in Sect. 1.3).
- **Qualitative examples** of correct and incorrect predictions to support the discussion.

**Performance summary.**

| Model | Validation mAP | Mean per-class F1 |
|---|---|---|
| A1 (SmallResNet) | 0.401 | 0.452 |
| A2 (Standard ResNet-18) | 0.448 | 0.480 |
| B1 (ConvNeXt-Small, IN-21k → 1k FT) | **0.907** | **0.891** |
| B2 (ViT-B/16, IN-21k → 1k FT) | 0.882 | 0.869 |

Transfer learning (B1, B2) more than doubles from-scratch performance. ConvNeXt-Small (B1) is the strongest classifier.

**Where the classifiers struggle.** The hardest classes depend on model strength:

- *From-scratch models (A1, A2)* are bottlenecked by **rare classes**. A1's worst three are cow (0.136), pottedplant (0.174), bicycle (0.182). A2's worst are pottedplant (0.085), cow (0.162), diningtable (0.182). With only 749 training images, the model never sees enough positives of rare classes to learn a usable score.
- *Transfer-learning models (B1, B2)* mostly solve the rare-class problem. B1 reaches 1.00 AP on aeroplane, cat, train, bird, sheep, with even cow at 0.976. The remaining hard classes are intrinsically difficult, **chair in particular**: 0.61 for B1 (its worst class) and 0.63 for B2 (second worst, after sofa at 0.62).

**Why chair is hardest for the strong models.** Two factors likely contribute:

1. **Label noise from partial annotation.** Chairs frequently appear in the background of images annotated for other objects (people at dining tables, sofas in living rooms), but the annotation guidelines do not always tag every chair in the scene. The model therefore sees many chair-present images without a chair label, pushing the learned score for chair down on true positives.
2. **High intra-class visual variation.** Dining chairs, office chairs, armchairs, stools, and deck chairs share little surface-level structure. Learning a single "chair" representation that generalizes across these is harder than for a more visually consistent class like aeroplane or bus, even with strong pretraining.

This pattern of intrinsic hardness, not rarity, is what makes chair an effective target for the segmentation adversarial attack in Sect. 3.2.


In [ ]:
def report_classifier(model, val_loader, name="model"):
    val_loss, val_map, aps, scores, targets = evaluate(model, val_loader)
    print(f"[{name}] validation loss = {val_loss:.4f}")
    print(f"[{name}] validation mAP  = {val_map:.4f}")
    classes_with_pos = [labels[i] for i in range(len(labels)) if targets[:, i].sum() > 0]
    ap_df = pd.Series(aps, index=classes_with_pos).sort_values(ascending=False)
    print(ap_df.to_string())
    thr, f1 = best_per_class_f1(scores, targets)
    print(f"[{name}] mean per-class F1 = {f1.mean():.4f}")
    return {"map": val_map, "loss": val_loss, "aps": aps, "thresholds": thr, "f1": f1, "scores": scores, "targets": targets}


In [ ]:
# Validation report for all classifiers.
# Prints validation loss, mAP, per-class AP, and per-class F1 at tuned thresholds.
report_A1 = report_classifier(model_a1, val_loader, name="Model A1 (SmallResNet from scratch)")
print()
report_A2 = report_classifier(model_a2, val_loader, name="Model A2 (Standard ResNet-18 from scratch)")
print()
report_B1 = report_classifier(model_tl, val_loader, name="Model B1 (ConvNeXt-Small IN-22k)")
print()
report_B2 = report_classifier(model_vit, val_loader, name="Model B2 (ViT-B/16 IN-1K)")

In [ ]:
# Qualitative examples on the validation set: image, ground-truth labels, predicted labels.
@torch.no_grad()
def show_predictions(model, df, transform, thresholds, n=6, seed=0, name="model"):
    rng = np.random.RandomState(seed)
    idxs = rng.choice(len(df), size=n, replace=False)
    model.eval().to(DEVICE)
    fig, axs = plt.subplots(1, n, figsize=(2.6 * n, 4))
    for j, k in enumerate(idxs):
        row = df.iloc[k]
        img = row["img"]
        pil = torchvision.transforms.functional.to_pil_image(img)
        x = transform(pil).unsqueeze(0).to(DEVICE)
        scores = torch.sigmoid(model(x)).cpu().numpy().squeeze(0)
        pred = (scores >= thresholds).astype(int)
        truth = row[labels].values.astype(int)
        true_set = [lab for lab, t in zip(labels, truth) if t == 1]
        pred_set = [lab for lab, p in zip(labels, pred) if p == 1]
        axs[j].imshow(img)
        axs[j].axis("off")
        title = "true: " + (", ".join(true_set) or "(none)")
        title += "\npred: " + (", ".join(pred_set) or "(none)")
        axs[j].set_title(title, fontsize=7)
    fig.suptitle(name)
    fig.tight_layout()
    plt.show()


show_predictions(model_a1, val_split_df, eval_tf, report_A1["thresholds"], name="Model A1 (SmallResNet)")
show_predictions(model_a2, val_split_df, eval_tf, report_A2["thresholds"], name="Model A2 (Standard ResNet-18)")
show_predictions(model_tl, val_split_df, eval_tf, report_B1["thresholds"], name="Model B1 (ConvNeXt-Small)")
show_predictions(model_vit, val_split_df, eval_tf, report_B2["thresholds"], name="Model B2 (ViT-B/16)")


## 1.9 Test-set predictions

We select the better-performing model, which is almost certainly Model B, run it on the test set, and write the binary multi-label predictions back into `test_df`. The helper `generate_submission` then converts `test_df` into the Kaggle CSV at the end of the notebook. The per-class thresholds tuned on the validation set are reused here. We optionally lower them slightly to bias the predictions toward recall, since the competition is scored on the Dice coefficient and under-predicting rare classes is more harmful than over-predicting them.


In [ ]:
@torch.no_grad()
def predict_classification_scores(model, df, transform):
    model.eval().to(DEVICE)
    ds = VOCClassificationDataset(df, transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE, non_blocking=True)
        all_scores.append(torch.sigmoid(model(x)).cpu().numpy())
    return np.concatenate(all_scores, axis=0)


# Select the best-performing model based on validation mAP.
models = {
    "Model A1 (SmallResNet)": (model_a1, report_A1),
    "Model A2 (Standard ResNet-18)": (model_a2, report_A2),
    "Model B1 (ConvNeXt-Small)": (model_tl, report_B1),
    "Model B2 (ViT-B/16)": (model_vit, report_B2),
}

best_name = max(models, key=lambda k: models[k][1]["map"])
best_model, best_report = models[best_name]
print(f"Best model: {best_name} with validation mAP = {best_report['map']:.4f}")

# Generate test predictions using the best model.
test_scores = predict_classification_scores(best_model, test_df, eval_tf)
test_preds = (test_scores >= best_report["thresholds"][None, :]).astype(int)
test_df.loc[:, labels] = test_preds
test_df.head()


# 2. Segmentation
The task is to assign one of 21 labels (20 PASCAL VOC object classes + background) to **every pixel** of every image. Unlike multi-label classification, segmentation is a *dense* prediction problem: for an input of shape `H x W x 3` the model must produce a class map of shape `H x W` (roughly 200,000 predictions per 448 px image).

The same data constraints as Section 1 apply (749 labelled images, long-tailed class distribution, single train/val split), but loss, metrics, and architecture choices differ because each pixel matters and the background class dominates.

**What we trained.** Three segmenters spanning three established architecture families, with progressively richer pre-training:

1. **U-Net** (Ronneberger et al., 2015). Symmetric encoder-decoder with skip connections at every spatial scale. Strong general-purpose baseline.
2. **DeepLabV3+** (Chen et al., 2018). Atrous (dilated) spatial pyramid pooling captures multi-scale context without resolution loss. A lightweight decoder upsamples back to the input.
3. **DINOv2 + multi-layer decoder** (Oquab et al., 2023). Self-supervised ViT backbone is frozen, only a small decoder learns the task-specific mapping.

| | Model A | Model B | Model C |
|---|---|---|---|
| Architecture | U-Net | DeepLabV3+ | DINOv2 + multi-layer decoder |
| Encoder | EfficientNet-B4 | EfficientNet-B4 | ViT-B/14 with registers (frozen) |
| Pre-training | ImageNet-1k (supervised) | ImageNet-1k (supervised) | LVD-142M (self-supervised) |
| Trainable params | ~19M (full) | ~22M (full) | ~4M (decoder only) |
| Input resolution | 384 x 384 | 384 x 384 | 448 x 448 |
| Best val mIoU | 0.4617 | 0.4384 | **0.7486** |

Central finding: Model C beats the EfficientNet-based baselines by +28 mIoU points. **On a 749-image dataset, the quality of the pre-trained features dominates the choice of decoder architecture.** Section 2.4 expands on the reasoning. Only Model C's training is included in this notebook. Models A and B were trained offline as references.

**What is the difference with a classification setup?**
Classification predicts a single image-level label vector (one output per image), whereas segmentation is a dense prediction problem requiring one class label per pixel — roughly 200,000 predictions per 448 px image.
This shifts every design decision. All three segmentation models (U-Net, DeepLabV3+, and DINOv2 + decoder) must preserve or recover spatial resolution through their architectures: U-Net via symmetric encoder-decoder skip connections, DeepLabV3+ via atrous spatial pyramid pooling, and our DINOv2 model via patch tokens reshaped into a 32 × 32 spatial grid and upsampled by a convolutional decoder — discarding the CLS token that a classification head would use. The loss also changes: instead of per-image Asymmetric Loss, we use Dice + weighted cross-entropy to handle per-pixel class imbalance (background covers ~70–80% of pixels).


**Alternatives considered.**

- *Vanilla U-Net trained from scratch.* No external pre-training. Section 1 already showed that 749 images is far below the data scale needed to learn useful features from scratch (Model A2 reached ~0.50 val mAP versus 0.85+ for transfer learning).
- *Different backbones for Models A and B.* `resnet50` instead of `efficientnet-b4` gave 1 to 2 mIoU points lower at the same epoch budget (Tan and Le, 2019).
- *Larger DINOv2 variants (ViT-L/14, ViT-g/14).* Strictly more parameters and FLOPs. ViT-L/14 reports a marginal 1 to 2 mIoU improvement in the DINOv2 paper but doubles inference time. We kept ViT-B/14 for the speed-vs-quality balance.


## 2.1 Data pipeline

Three differences from the classification pipeline (Sect. 1.1):

- **Square resize to 448 x 448.** ViT-B/14 requires input divisible by patch size 14. 448/14 = 32 patches per side (1024 tokens). Smaller resolutions discard spatial detail. Larger resolutions blow up VRAM. 448 is what the DINOv2 paper uses for ADE20K segmentation.
- **Joint image-and-mask transforms.** Spatial augmentations (flip, scale, shift, rotate, crop) must be applied to the image and mask in lockstep, otherwise pixel labels become misaligned. We use `albumentations`, which natively supports paired image-mask transforms via the `mask=` argument.
- **ImageNet mean/std normalization.** DINOv2 expects RGB in `[0, 1]` standardized with the ImageNet statistics. Test-time augmentation (Sect. 2.8) uses the same normalization.

**Augmentations** (training images only, used for Model C):

| Augmentation | Probability | Rationale |
|---|---|---|
| `HorizontalFlip` | 0.5 | PASCAL VOC has no canonical left-right orientation. |
| `ShiftScaleRotate` (±0.0625, ±0.20, ±15°) | 0.5 | Mild geometric perturbation. Border mode 0 fills out-of-frame pixels with zeros. |
| `RandomBrightnessContrast` / `HueSaturationValue` / `CLAHE` | one-of, total 0.5 | Photometric variation that does not change object semantics. |
| `GaussianBlur` / `GaussNoise` | one-of, total 0.2 | Mild capture-noise variation. Heavy blur would destroy small objects. |
| `CoarseDropout` (max 8 holes, 32x32 px) | 0.2 | Random rectangular masking. Forces the segmenter to use neighboring context rather than rely on a single discriminative region. |

**Train and validation split.** Random 80/20 split with `random_state=42`. PASCAL VOC segmentation masks contain multiple classes per image, so the multi-label stratification used in Section 1 does not directly apply. We keep a random split and verify per-class coverage manually after the split.


In [ ]:
class CFG:
    seed         = 42
    device       = "cuda" if torch.cuda.is_available() else "cpu"
    num_classes  = 21
    val_size     = 0.20
    image_size   = 448            # DINOv2 input (must be divisible by 14)
    batch_size   = 2
    epochs       = 23
    patience     = 10
    lr           = 1e-3
    weight_decay = 1e-4
    tta_scales   = (392, 448)

random.seed(CFG.seed); np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed); torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.benchmark = True
print(f"device={CFG.device}  classes={CFG.num_classes}")


VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog",
    "horse", "motorbike", "person", "pottedplant", "sheep", "sofa",
    "train", "tvmonitor",
]
assert len(VOC_CLASSES) == CFG.num_classes

def _voc_palette(n=21):
    palette = np.zeros((n, 3), dtype=np.uint8)
    for j in range(1, n):
        c = j
        for i in range(8):
            palette[j, 0] |= ((c >> 0) & 1) << (7 - i)
            palette[j, 1] |= ((c >> 1) & 1) << (7 - i)
            palette[j, 2] |= ((c >> 2) & 1) << (7 - i)
            c >>= 3
    return palette
VOC_PALETTE = _voc_palette()


MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def train_tf_light(size):
    """Used for Models A (U-Net) and B (DeepLabV3+) — matches what we ran."""
    return A.Compose([
        A.Resize(size, size),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, p=0.5),
        A.Normalize(MEAN, STD),
        ToTensorV2(),
    ])

def train_tf_strong(size):
    """Used for Model C (DINOv2). Stronger augmentation regularizes the small decoder."""
    return A.Compose([
        A.Resize(size, size),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.20, rotate_limit=15,
                           border_mode=0, p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(p=1.0),
            A.HueSaturationValue(p=1.0),
            A.CLAHE(p=1.0),
        ], p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.GaussNoise(var_limit=(5.0, 25.0), p=1.0),
        ], p=0.2),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                        min_holes=1, min_height=8, min_width=8,
                        fill_value=0, mask_fill_value=0, p=0.2),
        A.Normalize(MEAN, STD),
        ToTensorV2(),
    ])

def eval_tf(size):
    return A.Compose([A.Resize(size, size), A.Normalize(MEAN, STD), ToTensorV2()])


class VOCSegDataset(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = np.asarray(row["img"], dtype=np.uint8)
        mask = np.asarray(row["seg"], dtype=np.int64)
        out  = self.tf(image=img, mask=mask)
        return out["image"].float(), out["mask"].long()


train_idx, val_idx = train_test_split(
    np.arange(len(train_df)), test_size=CFG.val_size, random_state=CFG.seed
)
train_sub = train_df.iloc[train_idx].reset_index(drop=True)
val_sub   = train_df.iloc[val_idx].reset_index(drop=True)
print(f"train={len(train_sub)}  val={len(val_sub)}")


def make_loaders(image_size, batch_size, train_tf_fn):
    train_loader = DataLoader(
        VOCSegDataset(train_sub, train_tf_fn(image_size)),
        batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(
        VOCSegDataset(val_sub, eval_tf(image_size)),
        batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    return train_loader, val_loader



## 2.2 Loss function

Two pathologies dominate semantic segmentation training, and no single loss addresses both:

1. **Class imbalance.** In PASCAL VOC, background covers roughly 70 to 80% of pixels and several classes (`sheep`, `cow`, `pottedplant`) cover under 1%. Plain cross-entropy summed pixel-wise is dominated by background and converges to a model that predicts background nearly everywhere.
2. **Region-vs-pixel mismatch.** Cross-entropy optimizes per-pixel classification, but the metric we are graded on (Dice) scores the overlap of predicted and ground-truth *regions*. A model can have low cross-entropy and still produce fragmented, low-Dice masks.

We use a **combined Dice + Cross-Entropy loss**:

$$
\mathcal{L}(\hat{y}, y) = \tfrac{1}{2}\, \mathcal{L}_{\text{Dice}}(\hat{y}, y) + \tfrac{1}{2}\, \mathcal{L}_{\text{CE}}(\hat{y}, y)
$$

The Dice term directly optimizes region overlap. The CE term provides a smooth per-pixel gradient that stabilizes early training. Model C weights the CE term per class (inverse-square-root frequency, normalized to sum to the number of classes). Models A and B used unweighted CE.


In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, nc, smooth=1.0):
        super().__init__(); self.nc, self.s = nc, smooth
    def forward(self, logits, tgt):
        p = F.softmax(logits, 1)
        t = F.one_hot(tgt.clamp(min=0), self.nc).permute(0, 3, 1, 2).float()
        valid = (tgt >= 0).unsqueeze(1).float()           # ignore -1 pixels
        p, t = p * valid, t * valid
        i = (p * t).sum((0, 2, 3))
        d = p.sum((0, 2, 3)) + t.sum((0, 2, 3))
        return 1 - ((2 * i + self.s) / (d + self.s)).mean()

def compute_class_weights(masks_iter, num_classes):
    """Inverse-square-root frequency weights, normalized so they sum to num_classes."""
    counts = np.zeros(num_classes)
    for m in masks_iter:
        counts += np.bincount(np.asarray(m).ravel().clip(0, num_classes - 1),
                              minlength=num_classes)
    w = 1 / np.sqrt(counts / counts.sum() + 1e-6)
    w = w / w.sum() * num_classes
    return torch.tensor(w, dtype=torch.float32)

class_weights = compute_class_weights(train_sub["seg"], CFG.num_classes).to(CFG.device)
print("Class weights:", class_weights.cpu().numpy().round(3))


def make_loss_fn(use_class_weights=True, ignore_index=-1):
    """Build a combined Dice + CE loss. Models A,B used unweighted; Model C used weighted."""
    weight = class_weights if use_class_weights else None
    ce_fn   = nn.CrossEntropyLoss(weight=weight, ignore_index=ignore_index)
    dice_fn = DiceLoss(CFG.num_classes).to(CFG.device)
    def combo(logits, tgt):
        if isinstance(logits, tuple):                    # smp aux head support
            logits = logits[0]
        return 0.5 * ce_fn(logits, tgt) + 0.5 * dice_fn(logits, tgt)
    return combo

combo_loss = make_loss_fn(use_class_weights=True, ignore_index=-1)

## 2.3 Metrics

We follow the standard PASCAL VOC segmentation protocol and report:

- **mean Intersection-over-Union (mIoU)**, primary metric. Per class, $\text{IoU}_c = \text{TP}_c / (\text{TP}_c + \text{FP}_c + \text{FN}_c)$. Averaged across classes that appear in the validation set, with background treated as a class.
- **mean Dice (mDice)**, $\text{Dice}_c = 2\,\text{TP}_c / (2\,\text{TP}_c + \text{FP}_c + \text{FN}_c)$. Same ranking as mIoU but in the same units as the Kaggle scoring metric. Per-class F1 is numerically identical to per-class Dice for this confusion-matrix definition. We report both columns for convenience.


In [ ]:
def update_conf(conf, logits, masks, num_classes):
    """Accumulate a (num_classes, num_classes) confusion matrix in place."""
    if isinstance(logits, tuple): logits = logits[0]
    p = logits.argmax(1).flatten()
    t = masks.flatten()
    valid = (t >= 0) & (t < num_classes)
    p, t  = p[valid], t[valid]
    conf += torch.bincount(num_classes * t + p,
                           minlength=num_classes * num_classes
                           ).reshape(num_classes, num_classes).float()
    return conf

def metrics_from_conf(conf):
    tp = conf.diag()
    fp = conf.sum(0) - tp
    fn = conf.sum(1) - tp
    ok = conf.sum(1) > 0
    iou  = (tp / (tp + fp + fn + 1e-7))
    dice = (2 * tp / (2 * tp + fp + fn + 1e-7))
    return iou[ok].mean().item(), dice[ok].mean().item(), iou.cpu().numpy(), dice.cpu().numpy()

## 2.4 Model: DINOv2 ViT-B/14 (frozen) + multi-layer concat decoder

**Encoder.** DINOv2-with-registers ViT-B/14: 12 transformer blocks, hidden dimension 768, patch size 14. 86M parameters, all frozen (`requires_grad = False`). The backbone runs in `eval` mode and its forward pass is wrapped in `torch.no_grad()` so no activations are stored for backpropagation. This cuts training-time VRAM by roughly an order of magnitude and lets us train at 448 px on a single T4 GPU. Full fine-tuning would not fit.

**Decoder.** The decoder is the only trainable part. It follows the recipe described by Oquab et al. (2023) for ADE20K segmentation:

1. **Tap intermediate layers.** Forward hooks on transformer blocks 5 and 11 capture patch tokens at two depths. Early layers carry low-level texture cues. Late layers carry high-level semantics. Combining them is the segmentation analogue of a feature pyramid in a CNN.
2. **Drop CLS and register tokens.** The first `1 + num_register_tokens = 5` tokens are not spatial and are dropped before the patch tokens are reshaped into a `32 x 32` feature grid.
3. **Per-layer LayerNorm + 1x1 projection.** Each captured tensor is normalized and projected from 768 to 256 channels. LayerNorm matters because the two captured layers have very different activation scales. Without it, the deeper layer dominates the fusion.
4. **Concatenate-and-fuse.** The two 256-channel maps are concatenated to 512 channels and reduced back to 256 with a 1x1 convolution + BatchNorm + GELU.
5. **Refine.** Two 3x3 convolutions with BatchNorm and GELU recover local spatial structure lost in patch tokenization. Dropout 0.1 acts as regularization.
6. **Head + upsample.** A 1x1 convolution maps 256 channels to 21 class logits, then bilinear upsample to the input resolution (`448 x 448`).

**Why Model C beats the EfficientNet-B4 baselines by +28 mIoU points.** Two factors:

1. *Feature quality.* DINOv2 was trained self-supervised on roughly 100x more images than the ImageNet-1k supervised pretraining used by Models A and B (142M vs 1.3M). Its patch features transfer to dense prediction without fine-tuning (Oquab et al., 2023, Table 7). The frozen DINOv2 backbone with a small decoder reaches 0.75 mIoU. A fully fine-tuned U-Net or DeepLabV3+ on EfficientNet-B4 plateaus around 0.45 mIoU.
2. *Regularization through frozen weights.* Models A and B fine-tune ~20M parameters on 749 images, a parameter-to-image ratio of ~26,000:1. Model C trains only ~4M decoder parameters (5,000:1) while the 86M backbone parameters are frozen and cannot overfit. Training curves confirm this: A and B show moderate overfitting from epoch ~15, while Model C's validation loss stabilizes early and best mIoU is reached at epoch 21 of 23. Fine-tuning a 86M-parameter ViT with only 749 training images and ~4M decoder parameters would also be prone to catastrophic forgetting of the rich self-supervised features. The DINOv2 paper's ADE20K protocol uses a frozen backbone for the same reason.


In [ ]:
# --- Section 2.4 — DINOv2 model (self-contained, no CFG dependency) ---
BACKBONE_NAME    = "facebook/dinov2-with-registers-base"
BACKBONE_LAYERS  = (5, 11)
BACKBONE_PATCH   = 14
DECODER_CHANNELS = 256
DECODER_DROPOUT  = 0.1
NUM_CLASSES      = 21


class Decoder(nn.Module):
    def __init__(self, dim, num_classes, ch=256, n=4, drop=0.1):
        super().__init__()
        self.norms  = nn.ModuleList([nn.LayerNorm(dim) for _ in range(n)])
        self.projs  = nn.ModuleList([nn.Linear(dim, ch) for _ in range(n)])
        self.fuse   = nn.Sequential(
            nn.Conv2d(ch * n, ch, 1, bias=False), nn.BatchNorm2d(ch), nn.GELU())
        self.refine = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.GELU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm2d(ch), nn.GELU(),
            nn.Dropout2d(drop))
        self.head   = nn.Conv2d(ch, num_classes, 1)
    def forward(self, feats, h, w, size):
        outs = [proj(norm(f)).transpose(1, 2).reshape(-1, proj.out_features, h, w)
                for f, norm, proj in zip(feats, self.norms, self.projs)]
        x = self.fuse(torch.cat(outs, dim=1))
        return F.interpolate(self.head(self.refine(x)), size,
                             mode="bilinear", align_corners=False)


class DINOv2Segmenter(nn.Module):
    """DINOv2-with-registers ViT-B/14 (frozen) + multi-layer concat decoder."""
    def __init__(self, num_classes=NUM_CLASSES,
                 backbone_name=BACKBONE_NAME,
                 layers=BACKBONE_LAYERS,
                 patch_size=BACKBONE_PATCH,
                 channels=DECODER_CHANNELS,
                 dropout=DECODER_DROPOUT):
        super().__init__()
        self.layers = layers
        self.patch_size = patch_size
        self.backbone = AutoModel.from_pretrained(backbone_name)
        for p in self.backbone.parameters(): p.requires_grad = False
        self.backbone.eval()
        dim       = self.backbone.config.hidden_size                            # 768
        self.skip = 1 + getattr(self.backbone.config, "num_register_tokens", 0) # 5
        self.decoder = Decoder(dim, num_classes, channels, len(layers), dropout)

    def _feats(self, x):
        H, W = x.shape[-2:]
        captured = {}
        def make_hook(idx):
            def hook(m, inp, out):
                t = out[0] if isinstance(out, tuple) else out
                captured[idx] = t[:, self.skip:].detach()
            return hook
        hooks = [self.backbone.encoder.layer[i].register_forward_hook(make_hook(i))
                 for i in self.layers]
        with torch.no_grad():
            self.backbone(x)
        for h in hooks: h.remove()
        return [captured[i] for i in self.layers], \
               H // self.patch_size, W // self.patch_size, (H, W)

    def forward(self, x):
        feats, h, w, size = self._feats(x)
        return self.decoder(feats, h, w, size)

    def forward_with_grad(self, x):
        feats, h, w, size = self._feats(x)
        return self.decoder(feats, h, w, size)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = DINOv2Segmenter().to(DEVICE)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_train/1e6:.2f}M  (decoder only)")

In [ ]:
train_loader, val_loader = make_loaders(
    CFG.image_size, CFG.batch_size, train_tf_strong
)
print(f"train batches: {len(train_loader)}  val batches: {len(val_loader)}")

## 2.5 Regularization

With ~4M trainable parameters and 749 training images, controlled overfitting is the main concern. We combine:

- **Frozen backbone.** 86M backbone parameters are frozen and cannot overfit. Only the ~4M decoder parameters are trained.
- **Weight decay = 1e-4** (decoupled L2 penalty, applied via AdamW).
- **Dropout 0.1** in the decoder refine block.
- **BatchNorm** in the decoder fuse and refine convolutions. Acts as a mild regularizer because each example sees slightly different normalization statistics within the batch.
- **Aggressive data augmentation** (Sect. 2.1). With 749 images, augmentation is the main source of effective sample variety.


## 2.6 Training loop

The training loop for Model C uses:

- **AdamW optimizer** with `lr=1e-3`, `weight_decay=1e-4` (weight decay covered in Sect. 2.5).
- **OneCycleLR** with `max_lr=1e-3`, `pct_start=0.05` (5% warm-up). Combines warm-up with cosine decay. Lower LRs (1e-4) underfit the decoder. Higher LRs (3e-3) destabilize training.
- **Mixed-precision training** with `torch.amp.autocast(dtype=torch.float16)` and a `GradScaler`. Roughly halves VRAM and gives ~1.5x throughput.
- **Gradient clipping at norm 1.0**, applied after unscaling, to prevent destabilizing spikes.
- **Batch size 2**, constrained by VRAM at 448 x 448 with the ViT-B/14 forward.
- **23 epochs** with early-stopping patience 10. Validation mIoU plateaus after epoch 18 to 20.
- **Validation every epoch.** Tracks training loss, validation loss, mIoU, and mDice. The best-mIoU checkpoint is saved and reloaded at the end.


In [ ]:

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CFG.lr,
    total_steps=CFG.epochs * len(train_loader),
    pct_start=0.05)
scaler = GradScaler(device=CFG.device)

history = {"train_loss": [], "val_loss": [], "val_miou": [], "val_mdice": []}
best_miou, patience_left = -1.0, CFG.patience

for ep in range(1, CFG.epochs + 1):
    model.train(); model.backbone.eval()
    train_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f"ep{ep:02d}", leave=False):
        imgs, masks = imgs.to(CFG.device), masks.to(CFG.device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(CFG.device, dtype=torch.float16):
            loss = combo_loss(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    conf = torch.zeros(CFG.num_classes, CFG.num_classes, device=CFG.device)
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(CFG.device), masks.to(CFG.device)
            with autocast(CFG.device, dtype=torch.float16):
                logits = model(imgs)
            val_loss += combo_loss(logits, masks).item()
            update_conf(conf, logits, masks, CFG.num_classes)
    val_loss /= len(val_loader)
    miou, mdice, _, _ = metrics_from_conf(conf)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_miou"].append(miou)
    history["val_mdice"].append(mdice)

    print(f"ep{ep:3d}  train={train_loss:.4f}  val={val_loss:.4f}  "
          f"mIoU={miou:.4f}  mDice={mdice:.4f}")

    if miou > best_miou:
        best_miou, patience_left = miou, CFG.patience
        torch.save(model.state_dict(), "best_seg.pt")
        print(f"  ✓ best saved (mIoU={miou:.4f})")
    else:
        patience_left -= 1
        if patience_left == 0:
            print("  early stopping"); break
    torch.cuda.empty_cache()

model.load_state_dict(torch.load("best_seg.pt", map_location=CFG.device))
print(f"Loaded best model (mIoU={best_miou:.4f})")


def plot_seg_history(history, title=""):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="train loss", color="tab:blue")
    axes[0].plot(epochs, history["val_loss"],   label="val loss",   color="tab:orange")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
    axes[0].set_title("Loss"); axes[0].legend()
    axes[1].plot(epochs, history["val_miou"],  label="mIoU",  color="tab:green")
    axes[1].plot(epochs, history["val_mdice"], label="mDice", color="tab:purple")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("metric")
    axes[1].set_title("Validation metrics"); axes[1].legend()
    if title: fig.suptitle(title)
    fig.tight_layout(); plt.show()

plot_seg_history(history, title="Model C — DINOv2 + multi-layer decoder")

## 2.7 Evaluation and error analysis

For Model C on the validation set we produce:

- **Validation report** with mIoU, mDice, and per-class IoU and Dice (defined in Sect. 2.3, with `F1 == Dice` for this confusion-matrix evaluation).
- **Row-normalized confusion matrix** as a per-class recall view.
- **Qualitative side-by-side examples** (image, ground truth, prediction, overlay).
- **Summary table** comparing Model C with the offline-trained Models A and B.

Evaluation uses single-flip TTA at the training resolution. Sect. 2.8 uses multi-scale + flip TTA for the test set.

**Performance summary.** Model C reaches **mIoU = 0.7515** and **mDice = 0.8482** on the validation set with single-flip TTA, slightly above the no-TTA training-time best of 0.7486. Background (IoU 0.94) and cat (0.91) are the easiest classes. The remaining 19 classes range from 0.28 to 0.85.

**Where the segmenter struggles.** Two patterns dominate:

1. **Thin or fine-structured objects.** Bicycle (IoU 0.28, the worst class) and chair (0.45, second-worst) both have thin structural elements (frames, spokes, legs, slats) that fall below the 14-pixel patch grid used by ViT-B/14. With patches downsampling spatial resolution by 14x before the decoder, sub-patch structures are lost. The pattern repeats with motorbike (0.75).

2. **Indoor furniture and clutter.** Sofa (0.67), diningtable (0.68), tvmonitor (0.69), pottedplant (0.61), and chair are all indoor scene objects that overlap and occlude each other in cluttered images. Their boundaries are intrinsically ambiguous and partial annotation makes the labels themselves noisy.

**Why chair specifically.** The same factors discussed in Sect. 1.8 apply: chairs in the background of person and dining-table scenes are often unlabeled (label noise from partial annotation) and chair appearance varies widely between dining, office, and lounge variants (high intra-class variation). Combined with the thin-structure issue above, this is why chair drops from the strong-classifier mAP of 0.61 to 0.63 in Sect. 1.8 to a segmentation IoU of 0.45.


In [ ]:
class SEG_CFG:
    image_size   = 448
    batch_size   = 2
    epochs       = 23
    patience     = 10
    lr           = 1e-3
    weight_decay = 1e-4
    num_classes  = 21
    tta_scales   = (392, 448)
    device       = DEVICE

@torch.no_grad()
def evaluate_per_class(model, loader, num_classes=CFG.num_classes,
                       use_tta=True, name="Model C (DINOv2)"):
    model.eval()
    conf = torch.zeros(num_classes, num_classes, device=CFG.device)
    for imgs, masks in tqdm(loader, desc=f"eval {name}", leave=False):
        imgs, masks = imgs.to(CFG.device), masks.to(CFG.device)
        with autocast(CFG.device, dtype=torch.float16):
            logits = model(imgs)
            if use_tta:
                logits = logits + model(imgs.flip(-1)).flip(-1)
        update_conf(conf, logits, masks, num_classes)

    _, _, iou, dice = metrics_from_conf(conf)
    support = conf.sum(1).cpu().numpy().astype(int)
    df = pd.DataFrame({
        "support_pixels": support,
        "IoU":  iou.round(4),
        "Dice": dice.round(4),
        "F1":   dice.round(4),
    }, index=VOC_CLASSES)
    present = df["support_pixels"] > 0
    print(f"\n=== {name} ===")
    print(df.to_string())
    print(f"\nmIoU  = {df.loc[present, 'IoU'].mean():.4f}")
    print(f"mDice = {df.loc[present, 'Dice'].mean():.4f}")
    print(f"mF1   = {df.loc[present, 'F1'].mean():.4f}  (== mDice by definition)")
    return df, conf

df_c, conf_c = evaluate_per_class(model, val_loader, use_tta=True)


def plot_confusion_matrix(conf, classes=VOC_CLASSES, normalize="row", title=""):
    cm = conf.cpu().numpy().astype(float)
    if normalize == "row":
        cm = cm / (cm.sum(axis=1, keepdims=True) + 1e-9)
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm, vmin=0, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=90, fontsize=8)
    ax.set_yticklabels(classes, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("ground truth")
    ax.set_title(title or "Validation confusion matrix (row-normalized = recall)")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

plot_confusion_matrix(conf_c, title="Model C — validation confusion matrix")


@torch.no_grad()
def show_segmentation_predictions(model, df, image_size=CFG.image_size, *,
                                  n=6, seed=0, use_tta=True,
                                  name="Model C — DINOv2"):
    rng  = np.random.RandomState(seed)
    idxs = rng.choice(len(df), size=n, replace=False)
    palette = VOC_PALETTE
    tf = eval_tf(image_size)
    model.eval()

    fig, axs = plt.subplots(n, 4, figsize=(16, 4 * n))
    for r, k in enumerate(idxs):
        row = df.iloc[k]
        img = np.asarray(row["img"], dtype=np.uint8)
        gt  = np.asarray(row["seg"], dtype=np.int64)
        H, W = img.shape[:2]

        inp = tf(image=img)["image"].unsqueeze(0).float().to(CFG.device)
        with autocast(CFG.device, dtype=torch.float16):
            logits = model(inp)
            if use_tta:
                logits = logits + model(inp.flip(-1)).flip(-1)
        logits = F.interpolate(logits, size=(H, W), mode="bilinear", align_corners=False)
        pred   = logits.argmax(1).squeeze(0).cpu().numpy()

        gt_clip   = np.clip(gt,   0, 20)
        pred_clip = np.clip(pred, 0, 20)
        gt_rgb    = palette[gt_clip]
        pred_rgb  = palette[pred_clip]
        overlay   = (0.55 * img + 0.45 * pred_rgb).astype(np.uint8)

        present = sorted({c for c in np.unique(pred_clip) if c != 0} |
                         {c for c in np.unique(gt_clip)   if c != 0})
        legend  = ", ".join(VOC_CLASSES[c] for c in present)

        for c, (im, ttl) in enumerate(zip(
            [img, gt_rgb, pred_rgb, overlay],
            ["image", "ground truth", "prediction", "overlay"])):
            axs[r, c].imshow(im); axs[r, c].axis("off")
            if r == 0: axs[r, c].set_title(ttl)
        axs[r, 1].set_title(f"GT: {legend or '(empty)'}", fontsize=8)

    fig.suptitle(name, fontsize=12)
    fig.tight_layout(); plt.show()

show_segmentation_predictions(model, val_sub, n=6, seed=42,
                              name="Model C — DINOv2 — validation")


present = df_c["support_pixels"] > 0
mIoU_C, mDice_C = df_c.loc[present, "IoU"].mean(), df_c.loc[present, "Dice"].mean()

mIoU_A, mDice_A = 0.4617, None
mIoU_B, mDice_B = 0.4384, None

summary = pd.DataFrame({
    "Model A (U-Net + EfficientNet-B4)":      [mIoU_A, mDice_A],
    "Model B (DeepLabV3+ + EfficientNet-B4)": [mIoU_B, mDice_B],
    "Model C (DINOv2 frozen + decoder)":      [mIoU_C, mDice_C],
}, index=["mIoU", "mDice"]).round(4)
print(summary.to_string())

## 2.8 Test-set predictions

We use the best validation-mIoU checkpoint and run **test-time augmentation (TTA)** on every test image:

- Two scales: 392 px and 448 px (multi-scale inference is the standard +1 to 2 mIoU bump in segmentation).
- Two horizontal-flip variants: original and left-right flipped.

This gives 4 forward passes per image. Logits are upsampled to the original image resolution and averaged before the argmax. The averaged logits are slightly better calibrated than any single forward pass and produce smoother boundaries.


In [ ]:
@torch.no_grad()
def predict_test_with_tta(model, test_df_in, scales=CFG.tta_scales, do_flip=True):
    """Fill `test_df_in["seg"]` with predicted class maps using multi-scale + flip TTA."""
    model.eval()
    C = CFG.num_classes
    for idx, row in tqdm(test_df_in.iterrows(), total=len(test_df_in), desc="test-tta"):
        img  = np.asarray(row["img"], dtype=np.uint8)
        H, W = img.shape[:2]
        logits_acc = torch.zeros(C, H, W, device=CFG.device)
        for scale in scales:
            for flip in ([False, True] if do_flip else [False]):
                inp = eval_tf(scale)(image=img)["image"].unsqueeze(0).float().to(CFG.device)
                if flip: inp = inp.flip(-1)
                with autocast(CFG.device, dtype=torch.float16):
                    out = model(inp)
                if flip: out = out.flip(-1)
                logits_acc += F.interpolate(out, size=(H, W),
                                            mode="bilinear", align_corners=False).squeeze(0)
        test_df_in.at[idx, "seg"] = logits_acc.argmax(0).cpu().numpy().astype(np.int8)

predict_test_with_tta(model, test_df)

# Optional: visualize a few test predictions (no GT — the GT panel will look blank).
show_segmentation_predictions(model, test_df, n=4, seed=1,
                              name="Model C — DINOv2 — test predictions")

# Final Kaggle submission (uses the helper defined at the top of the notebook).
generate_submission(test_df)


# 3. Adversarial attack

Adversarial attacks add small, often imperceptible perturbations to an input image to make a model produce an incorrect prediction (Szegedy et al., 2014). The gradient-based pixel update used in FGSM was introduced by Goodfellow et al. (2015) and remains the building block of modern white-box attacks.

In this section we run two white-box attacks: an untargeted PGD attack on the classifier (3.1) and a targeted PGD attack on the segmenter (3.2). Both follow the optimization problem below.


**Adversarial perturbation.** Given a clean input $x$, we look for a small perturbation $\delta$ such that the adversarial input $x_{\text{adv}} = x + \delta$ fools the model while remaining imperceptible. Imperceptibility is enforced by an $L_p$ ball:

$$
\|\delta\|_p \leq \epsilon
$$

**Untargeted attack** (Sect. 3.1). Maximize the loss for the true class $y$:

$$
\delta^\star = \arg\max_{\|\delta\|_p \leq \epsilon} J(\theta, x + \delta, y)
$$

**Targeted attack** (Sect. 3.2). Minimize the loss for a chosen target class $y_{\text{target}}$:

$$
\delta^\star = \arg\min_{\|\delta\|_p \leq \epsilon} J(\theta, x + \delta, y_{\text{target}})
$$


## 3.1 Classification

**Setup.** White-box untargeted source-class suppression attack against Model B1 (ConvNeXt-Small, one of our top classifiers). The attacker has full access to the model's weights and gradients, justifying the white-box assumption (we trained the model ourselves). We attack the **person** class because it is the most frequent label in the dataset and has high baseline confidence, which makes a successful suppression more meaningful.

**What we tried first and why it failed.** We initially trained a small encoder-decoder perturbation network as a general-purpose adversary. It produced a 3-channel perturbation map passed through `tanh` and scaled by $\epsilon$. The loss combined a targeted false-positive BCE term, an $L_2$ perturbation penalty, a total-variation smoothness term, and a small preservation loss on non-target labels. We attacked Model A2 (ResNet-18 from scratch) because we expected the simplest model to be easiest to attack. Attack success rate plateaued near 0.5, mostly because the generator is data-hungry and the rarer source classes had too few examples to train on.

**What we chose instead.** A per-image white-box PGD attack on Model B1. PGD directly optimizes the perturbation pixels per image, sidestepping the data-scarcity problem and giving a much tighter result for assessing model vulnerability. Attacking the best classifier is also the more honest demonstration that a competitive model can still be fooled.


The model receives normalised ImageNet tensors, but perturbations are easier to constrain in normal pixel space [0, 1]. So we unnormalise, perturb, clamp, and normalise again.

In [ ]:
# ============================================================
# Normalisation utilities
# ============================================================

mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1).to(DEVICE)
std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1).to(DEVICE)

def normalize_img(x):
    """
    x: image tensor in [0, 1], shape (B, 3, H, W)
    returns ImageNet-normalised tensor.
    """
    return (x - mean) / std


def unnormalize_img(x):
    """
    x: ImageNet-normalised tensor, shape (B, 3, H, W)
    returns image tensor approximately in [0, 1].
    """
    return x * std + mean


def clamp_img(x):
    return torch.clamp(x, 0.0, 1.0)

### Untargeted source-class suppression attack

The source class we use is person, as is the most present label in the images and it has a good class performance. Essentially, what we do is:

Image with person class → PGD perturbation → push person class score down

In [ ]:
# ============================================================
# Freeze victim classifier: use best-performing model_tl
# ============================================================

victim_model = model_tl.to(DEVICE)
victim_model.eval()

for p in victim_model.parameters():
    p.requires_grad = False

print("Victim model: model_tl")
print("Victim model frozen:", all(not p.requires_grad for p in victim_model.parameters()))

### Perturbation constraint

We use the standard $L_\infty$ projection to bound the perturbation. After each gradient step, $\delta$ is clamped into $[-\epsilon, \epsilon]$ with $\epsilon = 8/255$. This single hard constraint keeps the perturbation imperceptible. No additional regularizers (L2, total variation) are added.


In [ ]:
# ============================================================
# Perturbation regularisation
# ============================================================

def total_variation_loss(delta):
    """
    Encourages smoother perturbations by penalising neighbouring pixel differences.
    """
    tv_h = torch.mean(torch.abs(delta[:, :, 1:, :] - delta[:, :, :-1, :]))
    tv_w = torch.mean(torch.abs(delta[:, :, :, 1:] - delta[:, :, :, :-1]))
    return tv_h + tv_w


def perturbation_norms(delta):
    """
    delta: perturbation in pixel space [0, 1], shape (B, 3, H, W)
    """
    with torch.no_grad():
        linf = delta.detach().abs().flatten(1).max(dim=1).values
        l2 = torch.sqrt(torch.sum(delta.detach().flatten(1) ** 2, dim=1))
    return linf, l2

In [ ]:
# ============================================================
# Direct PGD-style source-class untargeted attack
# ============================================================

def pgd_source_untargeted_attack(
    victim_model,
    x_norm,
    *,
    source_idx,
    epsilon=8/255,
    alpha=2/255,
    steps=20,
):
    """
    Direct white-box PGD attack for multi-label classification.

    Goal:
        For images containing the source class, reduce the victim model's
        confidence/logit for that source class.

    x_norm:
        ImageNet-normalised input tensor.

    Returns:
        x_adv_norm, delta
    """

    victim_model.eval()

    # Convert to pixel space [0, 1]
    x_raw = clamp_img(unnormalize_img(x_norm)).detach()

    # Initialise perturbation
    delta = torch.zeros_like(x_raw, requires_grad=True)

    for _ in range(steps):
        x_adv_raw = clamp_img(x_raw + delta)
        x_adv_norm = normalize_img(x_adv_raw)

        logits_adv = victim_model(x_adv_norm)
        source_logits = logits_adv[:, source_idx]

        # We minimise source logit to make the model stop predicting the source class.
        loss = source_logits.mean()

        grad = torch.autograd.grad(loss, delta)[0]

        # Gradient descent on source logit
        delta = delta - alpha * grad.sign()

        # Project back into L_inf epsilon-ball
        delta = torch.clamp(delta, -epsilon, epsilon)

        # Keep image valid after perturbation
        delta = torch.clamp(x_raw + delta, 0.0, 1.0) - x_raw

        delta = delta.detach().requires_grad_(True)

    x_adv_raw = clamp_img(x_raw + delta.detach())
    x_adv_norm = normalize_img(x_adv_raw)

    return x_adv_norm, delta.detach()

In [ ]:
# ============================================================
# Evaluate direct PGD source-class attack
# ============================================================

@torch.no_grad()
def _empty_cache_if_needed():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def evaluate_pgd_source_attack(
    victim_model,
    loader,
    *,
    source_idx,
    epsilon=8/255,
    alpha=2/255,
    steps=20,
    source_threshold=0.5,
):
    victim_model.eval()

    all_clean_scores = []
    all_adv_scores = []
    all_targets = []
    all_linf = []
    all_l2 = []

    for x_norm, y in loader:
        x_norm = x_norm.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True).float()

        # Clean predictions
        with torch.no_grad():
            clean_logits = victim_model(x_norm)
            clean_scores = torch.sigmoid(clean_logits)

        # PGD attack needs gradients, so not inside no_grad
        x_adv_norm, delta = pgd_source_untargeted_attack(
            victim_model,
            x_norm,
            source_idx=source_idx,
            epsilon=epsilon,
            alpha=alpha,
            steps=steps,
        )

        with torch.no_grad():
            adv_logits = victim_model(x_adv_norm)
            adv_scores = torch.sigmoid(adv_logits)
            linf, l2 = perturbation_norms(delta)

        all_clean_scores.append(clean_scores.cpu().numpy())
        all_adv_scores.append(adv_scores.cpu().numpy())
        all_targets.append(y.cpu().numpy())
        all_linf.append(linf.cpu().numpy())
        all_l2.append(l2.cpu().numpy())

    clean_scores = np.concatenate(all_clean_scores, axis=0)
    adv_scores = np.concatenate(all_adv_scores, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    linf = np.concatenate(all_linf)
    l2 = np.concatenate(all_l2)

    clean_source_scores = clean_scores[:, source_idx]
    adv_source_scores = adv_scores[:, source_idx]

    clean_detected_mask = clean_source_scores >= source_threshold

    if clean_detected_mask.sum() > 0:
        asr = np.mean(adv_source_scores[clean_detected_mask] < source_threshold)
    else:
        asr = 0.0

    confidence_drop = clean_source_scores - adv_source_scores

    clean_map, clean_aps = compute_map(clean_scores, targets)
    adv_map, adv_aps = compute_map(adv_scores, targets)

    return {
        "asr": float(asr),
        "num_eval_samples": int(len(clean_source_scores)),
        "num_clean_detected_source": int(clean_detected_mask.sum()),
        "mean_clean_source_score": float(clean_source_scores.mean()),
        "mean_adv_source_score": float(adv_source_scores.mean()),
        "mean_source_confidence_drop": float(confidence_drop.mean()),
        "median_source_confidence_drop": float(np.median(confidence_drop)),
        "clean_source_recall_on_subset": float(clean_detected_mask.mean()),
        "clean_map": float(clean_map),
        "adv_map": float(adv_map),
        "mean_linf": float(linf.mean()),
        "mean_l2": float(l2.mean()),
        "clean_scores": clean_scores,
        "adv_scores": adv_scores,
        "targets": targets,
        "clean_aps": clean_aps,
        "adv_aps": adv_aps,
    }


In [ ]:
# ============================================================
# Visual examples of PGD adversarial perturbations
# Source-class-specific untargeted attack
# ============================================================

def show_pgd_adversarial_examples(
    victim_model,
    df,
    transform,
    *,
    source_idx,
    source_class,
    thresholds,
    epsilon=8/255,
    alpha=2/255,
    steps=20,
    n=6,
    seed=0,
    amplify_delta=10.0,
    require_clean_detected=True,
    require_success=True,
):
    """
    Visualise direct PGD adversarial examples.

    For source-class-specific untargeted attack:
        - source class is present in the ground truth
        - clean model predicts source class as present
        - adversarial image makes source class disappear

    Important:
        Do NOT decorate this function with @torch.no_grad(),
        because PGD needs gradients.
    """

    rng = np.random.RandomState(seed)

    # Only consider images where the source class is present.
    candidate_df = df[df[source_class] == 1].reset_index(drop=True)

    if len(candidate_df) == 0:
        print(f"No images found with source class '{source_class}'.")
        return

    victim_model.eval()

    # Random order, but we may skip examples that do not satisfy conditions.
    idxs = rng.permutation(len(candidate_df))

    shown_examples = []

    for idx in idxs:
        if len(shown_examples) >= n:
            break

        row = candidate_df.iloc[idx]
        img_np = row["img"]
        pil = torchvision.transforms.functional.to_pil_image(img_np)

        x_norm = transform(pil).unsqueeze(0).to(DEVICE)
        y_true = row[labels].values.astype(int)

        # Clean prediction
        with torch.no_grad():
            clean_logits = victim_model(x_norm)
            clean_scores = torch.sigmoid(clean_logits).squeeze(0).cpu().numpy()

        # PGD adversarial attack
        # This needs gradients, so it must not be inside torch.no_grad().
        x_adv_norm, delta = pgd_source_untargeted_attack(
            victim_model,
            x_norm,
            source_idx=source_idx,
            epsilon=epsilon,
            alpha=alpha,
            steps=steps,
        )

        # Adversarial prediction
        with torch.no_grad():
            adv_logits = victim_model(x_adv_norm)
            adv_scores = torch.sigmoid(adv_logits).squeeze(0).cpu().numpy()

        source_threshold = thresholds[source_idx]

        clean_source_detected = clean_scores[source_idx] >= source_threshold
        attack_success = adv_scores[source_idx] < source_threshold

        if require_clean_detected and not clean_source_detected:
            continue

        if require_success and not attack_success:
            continue

        # Convert images back to pixel space for visualisation.
        x_raw = clamp_img(unnormalize_img(x_norm))
        x_adv_raw = clamp_img(unnormalize_img(x_adv_norm))

        x_raw_np = x_raw.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
        x_adv_np = x_adv_raw.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
        delta_np = delta.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()

        # Visualise perturbation around grey, amplified.
        delta_vis = np.clip(0.5 + amplify_delta * delta_np, 0, 1)

        clean_pred = (clean_scores >= thresholds).astype(int)
        adv_pred = (adv_scores >= thresholds).astype(int)

        true_set = [lab for lab, t in zip(labels, y_true) if t == 1]
        clean_set = [lab for lab, p in zip(labels, clean_pred) if p == 1]
        adv_set = [lab for lab, p in zip(labels, adv_pred) if p == 1]

        shown_examples.append({
            "x_raw_np": x_raw_np,
            "delta_vis": delta_vis,
            "x_adv_np": x_adv_np,
            "delta_np": delta_np,
            "true_set": true_set,
            "clean_set": clean_set,
            "adv_set": adv_set,
            "clean_source_score": clean_scores[source_idx],
            "adv_source_score": adv_scores[source_idx],
            "clean_source_detected": clean_source_detected,
            "attack_success": attack_success,
        })

    if len(shown_examples) == 0:
        print(
            "No examples matched the requested conditions. "
            "Try setting require_success=False or require_clean_detected=False."
        )
        return

    fig, axs = plt.subplots(len(shown_examples), 3, figsize=(10, 3.2 * len(shown_examples)))

    if len(shown_examples) == 1:
        axs = np.expand_dims(axs, axis=0)

    for row_i, ex in enumerate(shown_examples):
        axs[row_i, 0].imshow(ex["x_raw_np"])
        axs[row_i, 0].axis("off")
        axs[row_i, 0].set_title(
            "Original\n"
            f"true: {', '.join(ex['true_set']) or '(none)'}\n"
            f"pred: {', '.join(ex['clean_set']) or '(none)'}\n"
            f"{source_class}: {ex['clean_source_score']:.3f}",
            fontsize=8,
        )

        axs[row_i, 1].imshow(ex["delta_vis"])
        axs[row_i, 1].axis("off")
        axs[row_i, 1].set_title(
            f"PGD perturbation ×{amplify_delta:g}\n"
            f"||δ||∞={np.abs(ex['delta_np']).max():.4f}",
            fontsize=8,
        )

        axs[row_i, 2].imshow(ex["x_adv_np"])
        axs[row_i, 2].axis("off")
        axs[row_i, 2].set_title(
            "Adversarial\n"
            f"pred: {', '.join(ex['adv_set']) or '(none)'}\n"
            f"{source_class}: {ex['adv_source_score']:.3f}",
            fontsize=8,
        )

    fig.suptitle(
        f"Untargeted PGD attack on model_tl: suppress '{source_class}' prediction\n"
        f"ε={epsilon * 255:.0f}/255, α={alpha * 255:.0f}/255, steps={steps}",
        fontsize=12,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# Config for the classification PGD attack
SOURCE_CLASS = "person"           # most frequent class in the dataset
source_idx   = labels.index(SOURCE_CLASS)
EPSILON      = 8 / 255            # L_inf bound (~3% per channel, imperceptible)
ALPHA        = 2 / 255            # PGD step size
STEPS        = 20                 # PGD iterations
source_threshold = 0.5            # decision threshold for the source class (used by cell 99 plots)
# SEED is the global seed set in Sect. 0 (= 42)

# Section 2 (segmentation) overwrote `eval_tf` and `val_loader` with segmentation
# versions. Rebuild the classification versions here so the PGD attack runs on
# multi-label classification batches instead of segmentation masks. Downstream
# cells (show_pgd_adversarial_examples, show_gradcam_pgd_attack_examples) use
# these names too.
eval_tf = T.Compose([
    T.Resize(int(IMG_SIZE * 1.15)),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_loader = DataLoader(
    VOCClassificationDataset(val_split_df, eval_tf),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

# Run PGD over the validation set to get aggregate metrics for cell 99
pgd_metrics = evaluate_pgd_source_attack(
    victim_model,
    val_loader,
    source_idx=source_idx,
    epsilon=EPSILON,
    alpha=ALPHA,
    steps=STEPS,
)
print(f"ASR (source score < 0.5 after attack): {pgd_metrics['asr']:.3f}")
print(f"Mean source-score drop:   {pgd_metrics['mean_source_confidence_drop']:.4f}")
print(f"Median source-score drop: {pgd_metrics['median_source_confidence_drop']:.4f}")


In [ ]:
show_pgd_adversarial_examples(
    victim_model,
    val_split_df,
    eval_tf,
    source_idx=source_idx,
    source_class=SOURCE_CLASS,
    thresholds=report_B1["thresholds"] if "report_B1" in globals() else np.ones(NUM_CLASSES) * 0.5,
    epsilon=EPSILON,
    alpha=ALPHA,
    steps=STEPS,
    n=6,
    seed=SEED,
    amplify_delta=10.0,
    require_clean_detected=True,
    require_success=True,
)

### Explainability


In [ ]:
# ============================================================
# Grad-CAM for ConvNeXt source-class explainability
# ============================================================

class GradCAM:
    """
    Simple Grad-CAM implementation for CNN / ConvNeXt-style classifiers.

    The target layer must output a 4D feature map of shape:
        (B, C, H, W)

    For ConvNeXt-Small, we use one of the last spatial feature layers,
    e.g. "stages.3.blocks.2" or "stages.3.blocks.2.conv_dw".
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.fwd_hook = target_layer.register_forward_hook(self._save_activations)
        self.bwd_hook = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self.activations = out

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def remove_hooks(self):
        self.fwd_hook.remove()
        self.bwd_hook.remove()

    def __call__(self, x, class_idx):
        """
        x: normalized image tensor, shape (1, 3, H, W)
        class_idx: source class index
        """
        self.model.eval()
        self.model.zero_grad(set_to_none=True)

        logits = self.model(x)
        score = logits[:, class_idx].sum()

        score.backward(retain_graph=True)

        if self.activations is None or self.gradients is None:
            raise RuntimeError("Grad-CAM hooks did not capture activations/gradients.")

        if self.activations.ndim != 4:
            raise ValueError(
                f"Grad-CAM target layer must output a 4D tensor, "
                f"but got shape {self.activations.shape}."
            )

        # Global-average-pool gradients to obtain channel weights.
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)

        # Weighted sum of activations.
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        # Resize to input image size.
        cam = F.interpolate(
            cam,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        # Normalize to [0, 1].
        cam = cam.squeeze().detach()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        scores = torch.sigmoid(logits).detach().cpu().numpy()

        return cam.cpu().numpy(), scores

For Grad-CAM we hook the deepest feature map of ConvNeXt-Small (`stages.3.blocks.2`, shape `(1, 768, 7, 7)`). Channel-wise gradient weights produce a class-activation heatmap that highlights the regions the classifier relied on for the **person** prediction.


In [ ]:
# ============================================================
# Select Grad-CAM target layer for ConvNeXt-Small
# ============================================================

modules = dict(victim_model.named_modules())

# Recommended first choice:
# final ConvNeXt block before global pooling.
target_layer = modules["stages.3.blocks.2"]

print("Using Grad-CAM target layer: stages.3.blocks.2")

In [ ]:
# ============================================================
# Visualization helpers
# ============================================================

def overlay_heatmap_on_image(img_rgb, heatmap, alpha=0.45, cmap_name="jet"):
    """
    img_rgb: numpy image in [0, 1], shape (H, W, 3)
    heatmap: numpy heatmap in [0, 1], shape (H, W)
    """
    cmap = plt.get_cmap(cmap_name)
    heatmap_rgb = cmap(heatmap)[..., :3]
    overlay = (1 - alpha) * img_rgb + alpha * heatmap_rgb
    return np.clip(overlay, 0, 1)

In [ ]:
# ============================================================
# Make adversarial image using direct PGD
# ============================================================

def make_pgd_adversarial_image(
    victim_model,
    x_norm,
    *,
    source_idx,
    epsilon=8/255,
    alpha=2/255,
    steps=20,
):
    """
    Creates adversarial image using direct PGD source-class suppression.

    Returns:
        x_raw, delta, x_adv_raw, x_adv_norm
    """

    # PGD needs gradients, so do NOT wrap this in torch.no_grad().
    x_adv_norm, delta = pgd_source_untargeted_attack(
        victim_model,
        x_norm,
        source_idx=source_idx,
        epsilon=epsilon,
        alpha=alpha,
        steps=steps,
    )

    x_raw = clamp_img(unnormalize_img(x_norm))
    x_adv_raw = clamp_img(unnormalize_img(x_adv_norm))

    return x_raw, delta, x_adv_raw, x_adv_norm

In [ ]:
# ============================================================
# Grad-CAM comparison: clean vs PGD adversarial source explanation
# ============================================================

def show_gradcam_pgd_attack_examples(
    victim_model,
    df,
    transform,
    *,
    source_idx,
    source_class,
    thresholds,
    epsilon=8/255,
    alpha=2/255,
    steps=20,
    target_layer=None,
    n=4,
    seed=0,
    require_clean_detected=True,
    require_success=True,
):
    """
    Shows Grad-CAM before and after the PGD source-class suppression attack.

    For each example:
        - source class is present in the ground truth
        - clean model preferably predicts the source class
        - adversarial image preferably suppresses source score below threshold

    Grad-CAM is computed with respect to the source class logit.
    """

    rng = np.random.RandomState(seed)

    # Only consider images where the source class is present.
    candidate_df = df[df[source_class] == 1].reset_index(drop=True)

    if len(candidate_df) == 0:
        print(f"No images found with source class '{source_class}'.")
        return

    if target_layer is None:
        modules = dict(victim_model.named_modules())
        target_layer = modules["stages.3.blocks.2"]
        print("Using Grad-CAM target layer: stages.3.blocks.2")

    gradcam = GradCAM(victim_model, target_layer)

    idxs = rng.permutation(len(candidate_df))
    examples = []

    for idx in idxs:
        if len(examples) >= n:
            break

        row = candidate_df.iloc[idx]

        img_np = row["img"]
        pil = torchvision.transforms.functional.to_pil_image(img_np)
        x_norm = transform(pil).unsqueeze(0).to(DEVICE)

        y_true = row[labels].values.astype(int)

        # Create adversarial image using PGD.
        x_raw, delta, x_adv_raw, x_adv_norm = make_pgd_adversarial_image(
            victim_model,
            x_norm,
            source_idx=source_idx,
            epsilon=epsilon,
            alpha=alpha,
            steps=steps,
        )

        # Grad-CAM needs gradients, so do not use torch.no_grad() here.
        x_norm_gc = x_norm.detach().clone().requires_grad_(True)
        x_adv_norm_gc = x_adv_norm.detach().clone().requires_grad_(True)

        clean_cam, clean_scores = gradcam(x_norm_gc, source_idx)
        adv_cam, adv_scores = gradcam(x_adv_norm_gc, source_idx)

        clean_source_score = clean_scores[0, source_idx]
        adv_source_score = adv_scores[0, source_idx]

        source_threshold = thresholds[source_idx]

        clean_source_detected = clean_source_score >= source_threshold
        attack_success = adv_source_score < source_threshold

        if require_clean_detected and not clean_source_detected:
            continue

        if require_success and not attack_success:
            continue

        x_raw_np = x_raw.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
        x_adv_np = x_adv_raw.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
        delta_np = delta.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()

        perturb_energy = np.mean(np.abs(delta_np), axis=2)
        perturb_energy = perturb_energy / (perturb_energy.max() + 1e-8)

        clean_overlay = overlay_heatmap_on_image(x_raw_np, clean_cam)
        adv_overlay = overlay_heatmap_on_image(x_adv_np, adv_cam)

        clean_pred = (clean_scores.squeeze(0) >= thresholds).astype(int)
        adv_pred = (adv_scores.squeeze(0) >= thresholds).astype(int)

        true_set = [lab for lab, t in zip(labels, y_true) if t == 1]
        clean_set = [lab for lab, p in zip(labels, clean_pred) if p == 1]
        adv_set = [lab for lab, p in zip(labels, adv_pred) if p == 1]

        examples.append({
            "x_raw_np": x_raw_np,
            "x_adv_np": x_adv_np,
            "clean_overlay": clean_overlay,
            "adv_overlay": adv_overlay,
            "perturb_energy": perturb_energy,
            "delta_np": delta_np,
            "true_set": true_set,
            "clean_set": clean_set,
            "adv_set": adv_set,
            "clean_source_score": clean_source_score,
            "adv_source_score": adv_source_score,
        })

    gradcam.remove_hooks()

    if len(examples) == 0:
        print(
            "No examples matched the requested conditions. "
            "Try require_success=False or require_clean_detected=False."
        )
        return

    fig, axs = plt.subplots(len(examples), 5, figsize=(15, 3.2 * len(examples)))

    if len(examples) == 1:
        axs = np.expand_dims(axs, axis=0)

    for row_i, ex in enumerate(examples):
        axs[row_i, 0].imshow(ex["x_raw_np"])
        axs[row_i, 0].axis("off")
        axs[row_i, 0].set_title(
            "Clean image\n"
            f"{source_class}={ex['clean_source_score']:.3f}\n"
            f"pred: {', '.join(ex['clean_set']) or '(none)'}",
            fontsize=8,
        )

        axs[row_i, 1].imshow(ex["x_adv_np"])
        axs[row_i, 1].axis("off")
        axs[row_i, 1].set_title(
            "Adversarial image\n"
            f"{source_class}={ex['adv_source_score']:.3f}\n"
            f"pred: {', '.join(ex['adv_set']) or '(none)'}",
            fontsize=8,
        )

        axs[row_i, 2].imshow(ex["clean_overlay"])
        axs[row_i, 2].axis("off")
        axs[row_i, 2].set_title(
            f"Clean Grad-CAM\nfor '{source_class}'",
            fontsize=9,
        )

        axs[row_i, 3].imshow(ex["adv_overlay"])
        axs[row_i, 3].axis("off")
        axs[row_i, 3].set_title(
            f"Adversarial Grad-CAM\nfor '{source_class}'",
            fontsize=9,
        )

        axs[row_i, 4].imshow(ex["perturb_energy"], cmap="magma")
        axs[row_i, 4].axis("off")
        axs[row_i, 4].set_title(
            f"Perturbation energy\nmax |δ|={np.abs(ex['delta_np']).max():.4f}",
            fontsize=9,
        )

    fig.suptitle(
        f"Explainability of untargeted PGD attack: suppressing '{source_class}' prediction\n"
        f"ε={epsilon * 255:.0f}/255, α={alpha * 255:.0f}/255, steps={steps}",
        fontsize=13,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Run Grad-CAM comparison for PGD attack
# ============================================================

thresholds_tl = (
    report_B1["thresholds"]
    if "report_B1" in globals()
    else np.ones(NUM_CLASSES) * 0.5
)

modules = dict(victim_model.named_modules())
target_layer = modules["stages.3.blocks.2"]

show_gradcam_pgd_attack_examples(
    victim_model,
    val_split_df,
    eval_tf,
    source_idx=source_idx,
    source_class=SOURCE_CLASS,
    thresholds=thresholds_tl,
    epsilon=EPSILON,
    alpha=ALPHA,
    steps=STEPS,
    target_layer=target_layer,
    n=4,
    seed=SEED,
    require_clean_detected=True,
    require_success=True,
)

In [ ]:
# ============================================================
# Source confidence drop distribution
# ============================================================

clean_source_scores = pgd_metrics["clean_scores"][:, source_idx]
adv_source_scores = pgd_metrics["adv_scores"][:, source_idx]

score_drop = clean_source_scores - adv_source_scores

fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))

axs[0].hist(clean_source_scores, bins=20, alpha=0.7, label="clean")
axs[0].hist(adv_source_scores, bins=20, alpha=0.7, label="adversarial")
axs[0].axvline(source_threshold, color="red", linestyle="--", label="threshold")
axs[0].set_title(f"'{SOURCE_CLASS}' score distribution")
axs[0].set_xlabel("Source-class score")
axs[0].set_ylabel("# images")
axs[0].legend()

axs[1].hist(score_drop, bins=20, color="tab:purple", alpha=0.8)
axs[1].axvline(0, color="black", linestyle="--")
axs[1].set_title("Source score drop")
axs[1].set_xlabel("clean score - adversarial score")

axs[2].scatter(clean_source_scores, adv_source_scores, alpha=0.7)
axs[2].plot([0, 1], [0, 1], "k--")
axs[2].axhline(source_threshold, color="red", linestyle="--", label="threshold")
axs[2].axvline(source_threshold, color="red", linestyle=":", label="threshold")
axs[2].set_title("Clean vs adversarial source score")
axs[2].set_xlabel("Clean source score")
axs[2].set_ylabel("Adversarial source score")
axs[2].legend()

plt.tight_layout()
plt.show()

print(f"Mean source score drop: {score_drop.mean():.4f}")
print(f"Median source score drop: {np.median(score_drop):.4f}")
print(f"Fraction of images with decreased source score: {(score_drop > 0).mean():.4f}")
print(f"Fraction below threshold after attack: {(adv_source_scores < source_threshold).mean():.4f}")

### Results

The PGD attack was highly effective. Mean "person" confidence dropped by 0.9231 (median 0.9982) and every attacked image fell below the decision threshold after perturbation, meaning the model no longer detected "person" anywhere. This is a 100% untargeted success rate under the $L_\infty$ constraint used ($\epsilon = 8/255$). The Grad-CAM panels above show that the perturbation re-routes the classifier's attention away from the actual person region, confirming the attack mechanism is feature suppression rather than spurious activation elsewhere.


## 3.2 Segmentation

Same framework as Sect. 3.1, applied to Model C (DINOv2 + decoder, mIoU = 0.7486). Targeted white-box PGD: full access to weights and gradients, the white-box assumption is justified because we trained the model ourselves.

**Why person → chair.** Source class **person** because it is the most frequent class (207 of 749 images), giving the largest evaluation pool. Target class **chair** for three reasons:

1. Chair is among the model's weakest classes (per-class IoU 0.4521, second-worst after bicycle's 0.2831), so the model's chair representation is shallow and easier to push other inputs toward.
2. Chair is semantically and visually distant from person. A misclassification this large is dramatic.
3. In a real-world surveillance setting, being misclassified as a static, irrelevant object (chair) is a much more practical evasion goal than another vehicle or animal class.

**Attack method: per-image PGD.** For each image we:

1. Compute the clean prediction and build a **fake target mask** identical to the clean prediction except every **person** pixel is replaced by **chair**.
2. Initialize $\delta$ with uniform noise in $[-\epsilon, \epsilon]$ (random init avoids getting stuck in the first-step local optimum).
3. Run 250 PGD iterations. Each step computes a 10x pixel-weighted cross-entropy loss between the perturbed logits and the fake target mask, backpropagates to $\delta$, takes a sign-gradient step, and projects $\delta$ back into $[-\epsilon, \epsilon]$ (the $L_\infty$ ball).

The 10x weight on target pixels prevents the background (which dominates pixel count) from washing out the gradient. A custom `dinov2_forward_with_grad` is needed because the standard forward wraps the backbone in `torch.no_grad()` for efficiency, which would block the gradient path to the input.

**Why $\epsilon = 16/255$ for segmentation, $8/255$ for classification.** Classification only needs one logit to flip. Segmentation needs to flip thousands of spatially consistent pixel predictions simultaneously, a harder optimization that needs more budget. Both values are imperceptible to a human observer and are standard L_inf budgets for PGD on ImageNet-scale natural images (Madry et al., 2018).


In [ ]:
from albumentations.pytorch import ToTensorV2
import albumentations as A

def eval_tf(size):
    return A.Compose([A.Resize(size, size), A.Normalize(MEAN, STD), ToTensorV2()])

# Recreate val dataset
val_dataset = VOCSegDataset(val_sub, eval_tf(CFG.image_size))

VOC_CLASSES = [
    "background", "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow", "diningtable", "dog",
    "horse", "motorbike", "person", "pottedplant", "sheep", "sofa",
    "train", "tvmonitor"
]

def show_val_with_predictions(model, dataset, n=10, seed=42):
    model.eval()
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(dataset), n, replace=False)

    fig, axs = plt.subplots(n, 3, figsize=(12, n * 4))

    for row, i in enumerate(indices):
        img_tensor, true_mask = dataset[i]

        # Get model prediction
        with torch.no_grad():
            with autocast(CFG.device, dtype=torch.float16):
                pred = model(img_tensor.unsqueeze(0).to(CFG.device))
            pred_mask = pred.argmax(1).squeeze(0).cpu().numpy()

        # Denormalize image for display
        img_display = img_tensor.permute(1, 2, 0).numpy()
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_display = np.clip(img_display, 0, 1)

        # Get classes present in ground truth and prediction
        mask_np = true_mask.numpy()
        gt_classes = [VOC_CLASSES[c] for c in np.unique(mask_np) if c >= 0]
        pred_classes = [VOC_CLASSES[c] for c in np.unique(pred_mask) if c >= 0]

        axs[row, 0].imshow(img_display)
        axs[row, 0].set_title(f"Image {i}", fontsize=8)
        axs[row, 0].axis('off')

        axs[row, 1].imshow(mask_np, vmin=0, vmax=20, cmap='tab20')
        axs[row, 1].set_title(f"Ground Truth\n{', '.join(gt_classes)}", fontsize=7)
        axs[row, 1].axis('off')

        axs[row, 2].imshow(pred_mask, vmin=0, vmax=20, cmap='tab20')
        axs[row, 2].set_title(f"Prediction\n{', '.join(pred_classes)}", fontsize=7)
        axs[row, 2].axis('off')

    plt.tight_layout()
    plt.show()

show_val_with_predictions(model, val_dataset, n=10)

In [ ]:
# ── Class distribution in the FULL dataset (train + val) ────────────────────
# We use the full train_df for the adversarial attack — the segmentation model
# is already trained, so there is no reason to limit ourselves to val images.

from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

img_count = Counter()
pix_count = Counter()

for _, row in train_df.iterrows():
    mask = np.asarray(row["seg"]).ravel()
    for c in set(mask[mask > 0].tolist()):
        img_count[int(c)] += 1
        pix_count[int(c)] += int((mask == c).sum())

print(f"Full dataset: {len(train_df)} images\n")
print(f"{'Class':<16} {'Idx':>4}  {'# images':>9}  {'# pixels':>12}")
print('─' * 48)
for cls_idx, count in img_count.most_common():
    print(f"{VOC_CLASSES[cls_idx]:<16} {cls_idx:>4}  {count:>9}  {pix_count[cls_idx]:>12,}")

classes_sorted = [c for c, _ in img_count.most_common()]
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar([VOC_CLASSES[c] for c in classes_sorted],
       [img_count[c]   for c in classes_sorted], color='steelblue')
ax.set_ylabel("# images containing class")
ax.set_title("Class frequency of full dataset")
ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

best = img_count.most_common(1)[0][0]
print(f"\nMost common: '{VOC_CLASSES[best]}' ({img_count[best]} images)")


In [ ]:
from collections import Counter
import numpy as np

person_class = 15  # VOC index for "person"

total_images = len(train_df)
images_with_person = 0
total_person_pixels = 0

for _, row in train_df.iterrows():
    mask = np.asarray(row["seg"]).ravel()
    if person_class in mask:
        images_with_person += 1
        total_person_pixels += (mask == person_class).sum()

print(f"Total images in dataset       : {total_images}")
print(f"Images containing 'person'    : {images_with_person}")
print(f"Images WITHOUT 'person'       : {total_images - images_with_person}")
print(f"Total person pixels           : {total_person_pixels:,}")

**Run the attack.** Iterates the PGD attack over every image containing **person** in the training set (207 images).


In [ ]:
# =============================================================================================
# ----------------------------------- STEP 0. Configuration -----------------------------------
# =============================================================================================
SOURCE_CLASS = 15   # The class to attack. Index 15 in VOC_CLASSES is "person".
TARGET_CLASS = 9    # The class to fool the model into seeing. Index 9 is "chair".

# 16/255 ≈ 0.063 --> max amount any single pixel channel can be changed for it to be invisible to the human eye (pixel values are in [0,1])
# This is the radius of the allowed perturbation ball.
EPSILON      = 16. / 255

LR           = 2. / 255  # Step size of each PGD gradient update. Small steps allow finer convergence.
N_STEPS      = 250  # How many PGD iterations to run per image. More steps = the optimizer explores more of the allowed perturbation ball.

MAX_EVAL     = 207  # Attack is run on the first 200 images that contain a person.
MAX_PLOTS    = 20    # Only visualize the first 2 results to avoid too many graphs.

print(f"Attack : {VOC_CLASSES[SOURCE_CLASS]} -> {VOC_CLASSES[TARGET_CLASS]}")
print(f"Epsilon: {EPSILON*255:.0f}/255  |  Step: {LR*255:.2f}/255  |  Steps: {N_STEPS}")


# =============================================================================================
# -------------------- STEP 1. Gradient-enabled forward (float16 backbone) --------------------
# =============================================================================================
def dinov2_forward_with_grad(model, x):
    H, W = x.shape[-2:]
    with torch.enable_grad():
        with autocast(CFG.device, dtype=torch.float16):
            out = model.backbone(x, output_hidden_states=True)
    hidden = out.hidden_states
    feats  = [hidden[i + 1][:, model.skip:] for i in model.layers]
    return model.decoder(feats, H // model.patch_size, W // model.patch_size, (H, W))


# =============================================================================================
# ----------------------------------- STEP 2. Freeze model ------------------------------------
# =============================================================================================

model.eval()
for p in model.parameters():
    p.requires_grad_(False)


# =============================================================================================
# ------------------------------ STEP 3. Find source-class images -----------------------------
# =============================================================================================

full_dataset = VOCSegDataset(train_df.reset_index(drop=True), eval_tf(CFG.image_size))
all_src = []
for i, (_, row) in enumerate(train_df.iterrows()):
    if SOURCE_CLASS in np.asarray(row["seg"]).ravel():
        all_src.append(i)

src_indices = all_src
print(f"Sampled for attack: {len(src_indices)} images (All images containing {VOC_CLASSES[SOURCE_CLASS]})")


# =============================================================================================
# ---------------------------- STEP 4. Instance-specific PGD Attack ---------------------------
# =============================================================================================
orig_person_count = 0
adv_person_count = 0
failed_cases = []
for i, img_idx in enumerate(src_indices[:MAX_EVAL]):
    img_t, mask_t = full_dataset[img_idx]
    img_t = img_t.unsqueeze(0).to(CFG.device)   # (1, 3, H, W)
    mask_t = mask_t.unsqueeze(0).to(CFG.device) # (1, H, W)

    # We want to force the model to predict TARGET_CLASS ONLY where SOURCE_CLASS is present.
    # To do this, we get the clean prediction first, and then swap the target labels.
    with torch.no_grad():
        with autocast(CFG.device, dtype=torch.float16):
            clean_logits = model(img_t)


    clean_pred_tensor = clean_logits.argmax(1)
    ideal_target = clean_pred_tensor.clone()

    ideal_target[mask_t == SOURCE_CLASS] = TARGET_CLASS
    ideal_target[clean_pred_tensor == SOURCE_CLASS] = TARGET_CLASS

    delta = torch.empty_like(img_t).uniform_(-EPSILON, EPSILON)
    delta = delta.to(CFG.device).requires_grad_(True)

    # Run PGD
    for step in range(N_STEPS):
        adv_input = img_t + delta  # adds the current perturbation to the clean image
        logits = dinov2_forward_with_grad(model, adv_input)

        # Loss: cross entropy with our new ideal_target silhouette
        loss_matrix = F.cross_entropy(logits, ideal_target, reduction='none')
        # Give 10x higher weight to the pixels we are actively trying to attack
        weight = torch.ones_like(ideal_target, dtype=torch.float32)
        weight[ideal_target == TARGET_CLASS] = 10.0
        loss = (loss_matrix * weight).mean()
        loss.backward()

        # PGD update
        with torch.no_grad():
            delta -= LR * delta.grad.sign()

            delta.clamp_(-EPSILON, EPSILON)
            delta.grad.zero_()


    with torch.no_grad():
        adv_input = img_t + delta
        with autocast(CFG.device, dtype=torch.float16):
            adv_logits = model(adv_input)

        adv_pred = adv_logits.argmax(1).squeeze(0).cpu().numpy()
        clean_pred = clean_pred_tensor.squeeze(0).cpu().numpy()

    if SOURCE_CLASS in clean_pred:
        orig_person_count += 1       # orig_person_count accumulates how many images originally had "person" detected.
    if SOURCE_CLASS in adv_pred:
        adv_person_count += 1        # adv_person_count counts how many images still have "person" after the attack.

    if SOURCE_CLASS in adv_pred and SOURCE_CLASS in clean_pred:
        failed_cases.append({
            'img': denorm(img_t),
            'clean_pred': clean_pred,
            'adv_pred': adv_pred,
            'mask': mask_t.cpu().numpy().squeeze(0),
            'idx': img_idx
        })

    # Denormalize image for display
    def denorm(t):
        t = t.squeeze(0).cpu().permute(1, 2, 0).numpy()
        t = t * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        return np.clip(t, 0, 1)

    # Extract unique classes
    def get_class_names(mask):
        classes = np.unique(mask)
        names = [VOC_CLASSES[c] for c in classes if c > 0 and c < 21]
        text = ", ".join(names) if names else "Background"
        return "\\n".join(textwrap.wrap(text, width=20))

    if i < MAX_PLOTS:
        fig, axs = plt.subplots(1, 4, figsize=(16, 4))

        orig_img_disp = denorm(img_t)
        adv_img_disp = denorm(adv_input)

        gt_title = get_class_names(mask_t.cpu().numpy())
        pred_title = get_class_names(clean_pred)
        adv_title = get_class_names(adv_pred)

        # Visualization using VOC_PALETTE
        palette = VOC_PALETTE
        clean_rgb = palette[np.clip(clean_pred, 0, 20)]
        adv_rgb = palette[np.clip(adv_pred, 0, 20)]

        axs[0].imshow(orig_img_disp)
        axs[0].set_title(f"Original Image\\nGT: {gt_title}")
        axs[0].axis("off")

        axs[1].imshow(clean_rgb)
        axs[1].set_title(f"Original Seg\\nPred: {pred_title}")
        axs[1].axis("off")

        axs[2].imshow(adv_img_disp)
        axs[2].set_title("Adversarial Image")
        axs[2].axis("off")

        axs[3].imshow(adv_rgb)
        axs[3].set_title(f"Adversarial Seg\\nPred: {adv_title}")
        axs[3].axis("off")

        plt.tight_layout()
        plt.show()


# =============================================================================================
# ---------------------------------- STEP 5. Attack Summary -----------------------------------
# =============================================================================================
print("=" * 60)
print(f"Attack Summary for {len(src_indices[:MAX_EVAL])} images:")
print(f"Images predicting '{VOC_CLASSES[SOURCE_CLASS]}' originally: {orig_person_count}")
print(f"Images predicting '{VOC_CLASSES[SOURCE_CLASS]}' after attack: {adv_person_count}")
if orig_person_count > 0:
    print(f"Success Rate (person completely removed): {100 * (1 - adv_person_count / orig_person_count):.1f}%")
print("=" * 60)

### Results

Attack run on all 207 training-set images containing **person**:

- Images where the segmentation model originally predicted person: 198 of 207.
- Images where the model still predicts person after the attack: 5.
- **Attack success rate (ASR): up to 97.5%** across runs (above 90% consistently). Random-uniform $\delta$ initialization causes mild run-to-run variation since different starts explore different regions of the $L_\infty$ ball.

Failures cluster on images with very large or complex person regions where 250 steps were not enough to flip every pixel. Visually the attack is still effective on those images: the residual person pixels concentrate at the most visually distinctive parts (head, torso) where the model's person features are strongest, while the bulk of the region has already been flipped to chair.

The perturbed images are still fully recognizable to a human, guaranteed by the $L_\infty$ constraint ($\epsilon = 16/255$). This does not mean the model is unreliable under normal use. It performs well on natural inputs. But it cannot be trusted in security-critical applications where an adversary may craft inputs deliberately. The standard defense is **adversarial training**: generating adversarial examples during training and including them in the training batches so the model learns to resist them.

**Is this a realistic attack?** It is digital and white-box, so we assume the attacker has full access to weights and gradients. In practice deployed systems do not expose internals directly, but many are built on publicly released model families, and adversarial examples transfer across models trained on similar data. A white-box attack therefore provides a realistic upper bound on the vulnerability of a black-box system.


In [ ]:
# ── Visualize failed cases ────────────────────────────────────────────────────
print(f"\nFailed cases (person still detected after attack): {len(failed_cases)}")

if len(failed_cases) > 0:
    fig, axs = plt.subplots(len(failed_cases), 3, figsize=(12, len(failed_cases) * 4))
    if len(failed_cases) == 1:
        axs = axs[np.newaxis, :]

    for row, case in enumerate(failed_cases):
        clean_rgb = VOC_PALETTE[np.clip(case['clean_pred'], 0, 20)]
        adv_rgb   = VOC_PALETTE[np.clip(case['adv_pred'],   0, 20)]

        clean_classes = get_class_names(case['clean_pred'])
        adv_classes   = get_class_names(case['adv_pred'])

        axs[row, 0].imshow(case['img'])
        axs[row, 0].set_title(f"Image {case['idx']}\nOriginal", fontsize=8)
        axs[row, 0].axis('off')

        axs[row, 1].imshow(clean_rgb)
        axs[row, 1].set_title(f"Original pred\n{clean_classes}", fontsize=7)
        axs[row, 1].axis('off')

        axs[row, 2].imshow(adv_rgb)
        axs[row, 2].set_title(f"After attack (FAILED)\n{adv_classes}",
                               fontsize=7, color='red')
        axs[row, 2].axis('off')

    plt.suptitle("Failed cases: a person was still detected after perturbation",
                 fontsize=11, color='red')
    plt.tight_layout()
    plt.show()
else:
    print("No failed cases. Attack succeeded on all images")

# 4. Discussion

## 4.1 Classification

Transfer learning (ConvNeXt-Small and ViT-B/16, ImageNet-21k pre-trained) reached competitive mAP well above the from-scratch baselines. At 749 images, the choice of pretraining dataset is the dominant lever. The A1 vs A2 comparison shows that more capacity without external data only adds overfitting. With more time we would try test-time augmentation and a balanced sampling strategy for rare classes.

**Limitations.**
- Single 80/20 train/val split. Some classes have only a handful of validation images, so per-class AP carries high variance.
- Per-class F1 thresholds tuned on val leak a small amount of information.

## 4.2 Segmentation

Model C (DINOv2 frozen backbone) beat the EfficientNet-B4 baselines by +28 mIoU points. On 749 images, pre-trained feature quality dominates decoder architecture. Freezing 86M parameters also avoided catastrophic forgetting and let us train at 448 px on a single T4 GPU. The remaining weakness is the long-tailed class distribution: chair 0.45 IoU, bicycle 0.28.

**Limitations.**
- Random train/val split. A stratified split on per-class pixel coverage would better protect rare classes.
- The 14-pixel patch grid of ViT-B/14 destroys thin structures (Sect. 2.7). No CRF or boundary refinement.

## 4.3 Adversarial attacks

### 4.3.1 Classification
PGD reduced **person** confidence by 0.92 on average and pushed every attacked image below the decision threshold (100% untargeted success at $\epsilon = 8/255$). An earlier encoder-decoder generator approach plateaued at 50% ASR, which is why we switched to direct per-image PGD.

### 4.3.2 Segmentation
Targeted **person → chair** PGD achieved up to 97.5% ASR (5 of 198 failed) at $\epsilon = 16/255$. Failures concentrated on the most visually distinctive person regions (head, torso).

### Realism and defenses
Both attacks are white-box. This is not the typical deployment scenario, but adversarial examples transfer across models trained on similar data, so white-box attacks give a realistic upper bound. Adversarial training is the standard defense and would be the next step.

### Real-world takeaways
High in-distribution accuracy does not imply real-world reliability. Transfer learning from large, diverse pre-training sets is the biggest lever at our data scale, and self-supervised pre-training (DINOv2) beat supervised ImageNet pretraining.

**Limitations.**
- White-box only. No black-box transferability tests.
- No defenses tested. Adversarial training would be the next step but was beyond our compute budget.

## 4.4 Connection to lecture concepts

The classical computer-vision pipeline was hand-crafted feature extraction (SIFT, HOG) followed by a simple classifier (SVM). Transfer learning is structurally identical, a feature extractor followed by a classifier, except the feature extractor is now a deep network pre-trained on massive external data and the classifier is the small head fine-tuned on the task. The factor that made B1 (0.907 mAP) and B2 (0.882 mAP) so much stronger than from-scratch A1/A2 (0.40 to 0.45 mAP) is the same factor that made deep features replace SIFT: better representations, learned at a scale no single downstream task could provide.

Model C in segmentation extends this. DINOv2's self-supervised pre-training uses ~100x more images than ImageNet-21k, and its features are still richer despite using no labels. The result is +28 mIoU over fine-tuned U-Net and DeepLabV3+ on EfficientNet-B4. In our small-data regime (749 images), the choice of pre-trained feature extractor dominates every other architectural decision.


## References

[Szegedy et al., 2014] Szegedy, C., Zaremba, W., Sutskever, I., Bruna, J., Erhan, D., Goodfellow, I. J., and Fergus, R. *Intriguing Properties of Neural Networks*. International Conference on Learning Representations, 2014. Available at: https://arxiv.org/abs/1312.6199

[Goodfellow et al., 2015] Goodfellow, I. J., Shlens, J., and Szegedy, C. *Explaining and Harnessing Adversarial Examples*. International Conference on Learning Representations, 2015. Available at: https://arxiv.org/abs/1412.6572

[Ben-Baruch et al., 2021] Ben-Baruch, E., Ridnik, T., Zamir, N., Noy, A., Friedman, I., Protter, M., and Zelnik-Manor, L. *Asymmetric Loss For Multi-Label Classification*. ICCV, 2021. https://arxiv.org/abs/2009.14119

[Lin et al., 2017] Lin, T.-Y., Goyal, P., Girshick, R., He, K., and Dollár, P. *Focal Loss for Dense Object Detection*. ICCV, 2017. https://arxiv.org/abs/1708.02002

[Wu et al., 2020] Wu, T., Huang, Q., Liu, Z., Wang, Y., and Lin, D. *Distribution-Balanced Loss for Multi-Label Classification in Long-Tailed Datasets*. ECCV, 2020. https://arxiv.org/abs/2007.09654

[Everingham et al., 2010] Everingham, M., Van Gool, L., Williams, C. K. I., Winn, J., and Zisserman, A. *The Pascal Visual Object Classes (VOC) Challenge*. International Journal of Computer Vision, 88(2):303–338, 2010. https://doi.org/10.1007/s11263-009-0275-4

[Liu et al., 2022] Liu, Z., Mao, H., Wu, C.-Y., Feichtenhofer, C., Darrell, T., and Xie, S. *A ConvNet for the 2020s*. CVPR, 2022. https://arxiv.org/abs/2201.03545

[Ridnik et al., 2021] Ridnik, T., Ben-Baruch, E., Noy, A., and Zelnik-Manor, L. *ImageNet-21K Pretraining for the Masses*. NeurIPS Datasets and Benchmarks Track, 2021. https://arxiv.org/abs/2104.10972

[Sechidis et al., 2011] Sechidis, K., Tsoumakas, G., and Vlahavas, I. *On the Stratification of Multi-Label Data*. ECML PKDD, 2011. https://link.springer.com/chapter/10.1007/978-3-642-23808-6_10

[He et al., 2016] He, K., Zhang, X., Ren, S., and Sun, J. *Deep Residual Learning for Image Recognition*. CVPR, 2016. https://arxiv.org/abs/1512.03385

[Simonyan and Zisserman, 2015] Simonyan, K. and Zisserman, A. *Very Deep Convolutional Networks for Large-Scale Image Recognition*. ICLR, 2015. https://arxiv.org/abs/1409.1556

[Ioffe and Szegedy, 2015] Ioffe, S. and Szegedy, C. *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*. ICML, 2015. https://arxiv.org/abs/1502.03167

[He et al., 2015] He, K., Zhang, X., Ren, S., and Sun, J. *Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet Classification*. ICCV, 2015. https://arxiv.org/abs/1502.01852

[Dosovitskiy et al., 2021] Dosovitskiy, A., Beyer, L., Kolesnikov, A., Weissenborn, D., Zhai, X., Unterthiner, T., Dehghani, M., Minderer, M., Heigold, G., Gelly, S., Uszkoreit, J., and Houlsby, N. *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. ICLR, 2021. https://arxiv.org/abs/2010.11929
[Loshchilov and Hutter, 2019] Loshchilov, I. and Hutter, F. *Decoupled Weight Decay Regularization*. ICLR, 2019. https://arxiv.org/abs/1711.05101

[Srivastava et al., 2014] Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., and Salakhutdinov, R. *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*. JMLR, 15(56):1929-1958, 2014.
[Ronneberger et al., 2015] Ronneberger, O., Fischer, P., and Brox, T. *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI, 2015. https://arxiv.org/abs/1505.04597

[Chen et al., 2018] Chen, L.-C., Zhu, Y., Papandreou, G., Schroff, F., and Adam, H. *Encoder-Decoder with Atrous Separable Convolution for Semantic Image Segmentation*. ECCV, 2018. https://arxiv.org/abs/1802.02611

[Oquab et al., 2023] Oquab, M., Darcet, T., Moutakanni, T., et al. *DINOv2: Learning Robust Visual Features without Supervision*. arXiv, 2023. https://arxiv.org/abs/2304.07193

[Tan and Le, 2019] Tan, M. and Le, Q. V. *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks*. ICML, 2019. https://arxiv.org/abs/1905.11946
[Madry et al., 2018] Madry, A., Makelov, A., Schmidt, L., Tsipras, D., and Vladu, A. *Towards Deep Learning Models Resistant to Adversarial Attacks*. ICLR, 2018. https://arxiv.org/abs/1706.06083
